# DCGAN

## Imports

In [ ]:
#%matplotlib inline
import os
import re
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import time
import copy
import gc
import pandas as pd
from torchvision.transforms import InterpolationMode, AutoAugmentPolicy
from collections import defaultdict
from IPython.display import display, Image as Img
from PIL import Image
import cv2
from torch.utils.data import TensorDataset, Dataset, ConcatDataset, DataLoader
from openpyxl import load_workbook
from openpyxl.styles import Font
from torchvision.models import resnet34, vgg13, mobilenet_v2, squeezenet1_1, alexnet
from torchmetrics.classification import Accuracy
from torchvision.datasets import ImageFolder
from torchvision.transforms.functional import to_pil_image
from pytorch_pretrained_gans import make_gan
from pytorch_pretrained_biggan.utils import one_hot_from_names, truncated_noise_sample
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore
import torch.nn.functional as Functional
from torch.optim.lr_scheduler import CosineAnnealingLR


# Set random seed for reproducibility
manualSeed = 999
#manualSeed = random.randint(1, 10000) # use if you want new results
print("Random Seed: ", manualSeed)
random.seed(manualSeed)
torch.manual_seed(manualSeed)
torch.use_deterministic_algorithms(True) # Needed for reproducible results

print(torch.cuda.is_available())         # True
print(torch.version.cuda)                # '11.8'
print(torch.cuda.get_device_name(0))     # 'NVIDIA GeForce RTX 4070 SUPER'

print(torch.backends.cudnn.version())    # 8700
print(torch.backends.cudnn.enabled)      # True

# Number of GPUs available. Use 0 for CPU mode.
NGPU = 1
device = torch.device("cuda:0" if (torch.cuda.is_available() and NGPU > 0) else "cpu")

## Parameters

In [ ]:
# Root directory for dataset
# use of os.path.join to maximize campatibility beetween different Operating Systems
IMAGENETTE = "imagenette2" 
IMAGEWOOF = "imagewoof2"
DATABASES = [IMAGENETTE, IMAGEWOOF]
GENERATED = os.path.join("data", "generated")
datarootImagenette = os.path.join("data", "databases", IMAGENETTE)
datarootImagewoof =  os.path.join("data", "databases", IMAGEWOOF)
path_datarootImagewoofTrain = os.path.join(datarootImagewoof, "train")
path_datarootImagenetteTrain = os.path.join(datarootImagenette, "train")
path_datarootImagewoofVal = os.path.join(datarootImagewoof, "val")
path_datarootImagenetteVal = os.path.join(datarootImagenette, "val")
paths_dataroot = {IMAGENETTE: {'train': path_datarootImagenetteTrain, 'val': path_datarootImagenetteVal}, IMAGEWOOF: {'train': path_datarootImagewoofTrain, 'val': path_datarootImagewoofVal}}

PLOTS = os.path.join(GENERATED, "plots")
MODELS = os.path.join(GENERATED, "models")
MODELS_CLASSIFIERS = os.path.join(MODELS, "classifiers")
TRANSFORMED_DATASET = os.path.join(GENERATED, "transformed_datasets")
NUM_IMAGES_FOR_STEP = 32
ROUNDS_PER_SAVE_IMAGES = 1
ROUNDS_PER_SAVE_MODELS = 300
NUM_EPOCHS = 300
MEAN=[0.485, 0.456, 0.406]
STD=[0.229, 0.224, 0.225]

# Number of channels in the training images. For color images this is 3
NC = 3

# Beta1 hyperparameter for Adam optimizers
beta1 = 0.5

# Learning rates values
LEARNING_RATES = [
    #0.0001,
    0.0002,
    #0.0003,
    0.0004,
    #0.0005,
    0.0006,
    #0.0007,
    #0.0008,
    #0.0009,
    #0.0010,
]

NZ_VALUES = [
    100,
]

IMG_SIZES = [
    64, 
    #128, 
    #256
]

# Batch size during training
BATCH_SIZE = {
    64: 256,
    128: 256,
    256: 128,
}

TRAIN_DCGAN = False

if TRAIN_DCGAN:
    SHOW_EXAMPLE_IMAGES_DATASET = True
    CREATE_DATALOADERS = True
    CREATE_MODELS = True
    TRAIN_MODELS = True
    PLOT_LOSS = True
    PLOT_ACCURACY_EVOLUTION = True
    SHOW_PROGRESS_IMAGES = True
    SHOW_FINAL_RESULTS = True
    PLOT_TIME_TRAINED = True
else:
    SHOW_EXAMPLE_IMAGES_DATASET = False
    CREATE_DATALOADERS = False
    CREATE_MODELS = False
    TRAIN_MODELS = False
    PLOT_LOSS = False
    PLOT_ACCURACY_EVOLUTION = False
    SHOW_PROGRESS_IMAGES = False
    SHOW_FINAL_RESULTS = False
    PLOT_TIME_TRAINED = False


In [ ]:
# Ignorar por extensión
ignored_exts = {'.jpeg', '.pdf', '.pt', '.txt', 'png', '.tgz'}

# Ignorar por nombre exacto
ignored_names = {'.gitkeep'}

# Regex para ignorar archivos tipo n######## (como WordNet IDs)
wnid_pattern = re.compile(r'^n\d{8}$')

def print_tree(start_path, prefix=""):
    items = sorted(os.listdir(start_path))
    for item in items:
        path = os.path.join(start_path, item)

        # Ignorar por nombre exacto
        if item in ignored_names:
            continue

        # Ignorar por patrón tipo "n########"
        name_no_ext = os.path.splitext(item)[0]
        if wnid_pattern.fullmatch(name_no_ext):
            continue

        # Ignorar por extensión
        if os.path.isfile(path):
            ext = os.path.splitext(item)[1].lower()
            if ext in ignored_exts:
                continue
            print(f"{prefix}└── {item}")
        elif os.path.isdir(path):
            print(f"{prefix}├── {item}")
            print_tree(path, prefix + "│   ")

# Ruta raíz
root = "data"
print(f"/{root}")
print_tree(root)

Imagenette is a subset of 10 easily classified classes from Imagenet (tench, English springer, cassette player, chain saw, church, French horn, garbage truck, gas pump, golf ball, parachute).

Imagewoof is a subset of 10 classes from Imagenet that aren't so easy to classify, since they're all dog breeds. The breeds are: Australian terrier, Border terrier, Samoyed, Beagle, Shih-Tzu, English foxhound, Rhodesian ridgeback, Dingo, Golden retriever, Old English sheepdog.

In [ ]:
# IMAGENETTE
classifierImagenette = {
    "tench": "n01440764", # Type of fish
    "English springer": "n02102040", # Type of dog
    "cassette player": "n02979186", 
    "chain saw": "n03000684",
    "church": "n03028079",
    "French horn": "n03394916",
    "garbage truck": "n03417042",
    "gas pump": "n03425413",
    "golf ball": "n03445777",
    "parachute": "n03888257"
}

classifierImagenette_inv = {valor: clave for clave, valor in classifierImagenette.items()}

id_to_index_imagenette = {
    imagenet_id: idx for idx, imagenet_id in enumerate(sorted(classifierImagenette.values()))
}

# IMAGEWOOF
classifierImagewoof = {
    "Australian terrier": "n02086240", 
    "Border terrier": "n02087394", 
    "Samoyed": "n02088364", 
    "Beagle": "n02089973",
    "Shih-Tzu": "n02093754",
    "English foxhound": "n02096294",
    "Rhodesian ridgeback": "n02099601",
    "Dingo": "n02105641",
    "Golden retriever": "n02111889",
    "Old English sheepdog": "n02115641"
}

classifierImagewoof_inv = {valor: clave for clave, valor in classifierImagewoof.items()}

id_to_index_imagewoof = {
    imagenet_id: idx for idx, imagenet_id in enumerate(sorted(classifierImagewoof.values()))
}


# generalization
classifierFromName ={
    IMAGENETTE: classifierImagenette,
    IMAGEWOOF: classifierImagewoof
}

# generalization
classifierFromNameInv ={
    IMAGENETTE: classifierImagenette_inv,
    IMAGEWOOF: classifierImagewoof_inv
}


## Show examples of Images Imagenette and Imagewoof
Functions to make the dataset visuabel and make a better understanding of the data we are gonna use

In [ ]:
def show_images_dataset(database):
    if SHOW_EXAMPLE_IMAGES_DATASET:
        if database == IMAGENETTE:
            dataroot = datarootImagenette
            folderClassifier = classifierImagenette
        elif database == IMAGEWOOF:
            dataroot = datarootImagewoof
            folderClassifier = classifierImagewoof

        images = []
        class_names = []
        train_samples = []
        val_samples = []
        titles = []

        for key in folderClassifier:
            first_image_path = None
            titles.append(key)
            # Count train samples
            train_folder = os.path.join(dataroot, "train", folderClassifier[key])
            sample_count_train = len([x for x in os.listdir(train_folder) if x.lower().endswith(('jpg', 'png', 'jpeg'))])
            train_samples.append(sample_count_train)

            # Count validation samples
            val_folder = os.path.join(dataroot, "val", folderClassifier[key])
            sample_count_val = len([x for x in os.listdir(val_folder) if x.lower().endswith(('jpg', 'png', 'jpeg'))])
            val_samples.append(sample_count_val)

            class_names.append(key)

            # Find the first image in the training folder
            if not first_image_path:
                for file_name in os.listdir(train_folder):
                    if file_name.lower().endswith(('.jpg', '.png', '.jpeg')):
                        first_image_path = os.path.join(train_folder, file_name)
                        break

            if first_image_path and len(images) < 10:
                image = Image.open(first_image_path).convert('RGB')
                image = transforms.CenterCrop(min(image.size))(image)
                image = image.resize((224, 224), Image.LANCZOS)
                images.append(image)

        num_images = len(images)
        num_columns = 5
        num_rows = 2

        fig, axes = plt.subplots(num_rows, num_columns, figsize=(15, 7), constrained_layout=True)

        for i, ax in enumerate(axes.flat):
            if i < num_images:
                ax.imshow(images[i])
                ax.set_title(titles[i], fontsize=20)
                ax.axis('off')
            else:
                ax.axis('off')

        plt.savefig(os.path.join(PLOTS, f'{database}_samples.png'), bbox_inches='tight')
        plt.show()

        data = {
            'Dataset': ['Training', 'Validation']
        }

        total_train = sum(train_samples)
        total_val = sum(val_samples)

        for i, class_name in enumerate(class_names):
            data[class_name] = [train_samples[i], val_samples[i]]

        data['Total'] = [total_train, total_val]

        df = pd.DataFrame(data)
        grand_total = total_train + total_val
        df.loc[len(df)] = ['Overall Total'] + [''] * (len(df.columns) - 2) + [grand_total]

        df.to_excel(os.path.join(PLOTS, f'{database}_samples.xlsx'), index=False)
        display(df)


def most_common(lst):
    return max(set(lst), key=lst.count)

def get_image_dimension(path):
    im = cv2.imread(path)
    h, w, _ = im.shape
    return h, w

def show_dimensions_dataset(dataroot, folderClassifier, saveFile=None):
    imagesSize = []
    # Itera a través de cada categoría en el clasificador
    for key in folderClassifier:
        image_path = None
        folder_path = os.path.join(dataroot, "train", folderClassifier[key])
        # Save the dimension of every photo
        for file_name in os.listdir(folder_path):
            if file_name.endswith(('.jpg', '.png', '.jpeg', '.JPG', '.PNG', '.JPEG')):
                image_path = os.path.join(folder_path, file_name)
                imagesSize.append(get_image_dimension(image_path))


    widths, heights = zip(*imagesSize)
    
    # Min, max, most common and average widths
    dataWidths = [min(widths), max(widths), most_common(widths), round(sum(widths)/len(widths))]
    print(f"Width:\n -Lowest:  {dataWidths[0]}, Highest:  {dataWidths[1]}, Most common: {dataWidths[2]}, Average: {dataWidths[3]}")

    # Min, max, most common and average heights
    dataHeights = [min(heights), max(heights), most_common(heights), round(sum(heights)/len(heights))]
    print(f"Height:\n -Lowest:  {dataHeights[0]}, Highest:  {dataHeights[1]}, Most common: {dataHeights[2]}, Average: {dataHeights[3]}")
    
    # Guardar números y datos adicionales como XLSX (formato horizontal)
    if saveFile is not None:
        df = pd.DataFrame({
            'Lowest': [dataWidths[0], dataHeights[0]],
            'Highest': [dataWidths[1], dataHeights[1]],
            'Most common': [dataWidths[2], dataHeights[2]],
            'Average': [dataWidths[3], dataHeights[3]]
        }, index=['Width', 'Height'])

        df.to_excel(saveFile + ".xlsx")
        print("Dimensions saved as Excel file.")


    # Create the scatter plot
    fig = plt.figure(figsize=(10, 6))
    plt.xscale('log')  # Logarithmic scale for x-axis
    plt.yscale('log')  # Logarithmic scale for y-axis
    plt.scatter(widths, heights, marker='o', s=100)  # s controls the size of points
    plt.title("Image Dimensions (log scale)")
    plt.xlabel("Width (pixels) (log scale)")
    plt.ylabel("Height (pixels) (log scale)")
    
    # Custom formatter to avoid scientific notation on both axes
    ax = plt.gca()
    ax.xaxis.set_major_formatter(FuncFormatter(lambda val, pos: f'{val:.0f}'))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda val, pos: f'{val:.0f}'))

    # Annotate each point with its width and height
    for i, (w, h) in enumerate(imagesSize):
        plt.annotate(f"({w}, {h})", (w, h), textcoords="offset points", xytext=(5, 5), ha='center')

    plt.grid(True)
    if saveFile is not None:
        plt.savefig(saveFile+".png")
        print("Saved dimensions")
    plt.show()
    

def load_dimensions_dataset(database):
    """
    Load or compute dimensions dataset.

    Parameters:
        output_path (str): Path to save/load the output file.
        dataroot (str): Root directory of the dataset.
        folderClassifier (dict): Dictionary mapping class names to folder names.

    Returns:
        None
    """
    output_path = os.path.join(PLOTS, f"{database}_dimension_distribution")

    if database == IMAGENETTE:
        dataroot, folderClassifier = datarootImagenette,classifierImagenette
    elif database == IMAGEWOOF:
        dataroot, folderClassifier = datarootImagewoof,classifierImagewoof
    
    if not os.path.exists(output_path+".png"):
        # Compute and save dimensions dataset
        fig = show_dimensions_dataset(dataroot, folderClassifier, saveFile=output_path)
    else:
        # Load the precomputed dimensions dataset from Excel
        df = pd.read_excel(output_path + ".xlsx", index_col=0)

        dataWidths = df.loc['Width']
        dataHeights = df.loc['Height']

        print(f"Width:\n -Lowest: {dataWidths['Lowest']}, Highest: {dataWidths['Highest']}, "
            f"Most common: {dataWidths['Most common']}, Average: {dataWidths['Average']}")
        print(f"Height:\n -Lowest: {dataHeights['Lowest']}, Highest: {dataHeights['Highest']}, "
            f"Most common: {dataHeights['Most common']}, Average: {dataHeights['Average']}")

        display(Img(filename=output_path + ".png"))


### imaginette 
#### Image examples
Show images of every type of imaginette

In [ ]:
show_images_dataset(IMAGENETTE)

#### Distribution
Distribution of the images dimensions to see how the size looks like (A good idea to consider to wich size should be resized)

In [ ]:
load_dimensions_dataset(IMAGENETTE)

### imagewoof 
#### Image examples

Show all the images of imagewoof

In [ ]:
show_images_dataset(IMAGEWOOF)

#### Distribution

In [ ]:
load_dimensions_dataset(IMAGEWOOF)

## Training DCGANS

### Create a dataset of the images.


In [ ]:
def show_images_grid(dataset, dataloader, titlePlot):# Plot some training images
    
    # Número total de imágenes en el dataset
    total_images = len(dataset)
    print(f"Total de imágenes en el dataset: {total_images}")

    # Número de batches por epoch en el dataloader
    batches_per_epoch = len(dataloader)
    print(f"Número de batches por epoch: {batches_per_epoch}")

    real_batch = next(iter(dataloader))
    plt.figure(figsize=(8,8))
    plt.axis("off")
    plt.title(titlePlot)
    plt.imshow(np.transpose(vutils.make_grid(real_batch[0][:64], padding=2, normalize=True).cpu(),(1,2,0)))
    plt.show()


We use a class to create the dataset transforming the images when we create the dataset instead of when we iterate with them. Doing this we only transform the images once as we will train with the same dataset multiple models. The other way around the images where getting transformed every time we train a different model with the same dataset. 

In [ ]:
class PreloadedImageFolder(Dataset):
    def __init__(self, root, label, transform=None):
        super().__init__()
        self.root = root
        self.transform = transform
        self.preloaded_data = []
        self.label = torch.tensor(label)  # Guardar la etiqueta de la clase
        
        # Obtener solo archivos de imagen en la carpeta
        self.samples = [os.path.join(root, f) for f in os.listdir(root) if f.lower().endswith(('jpg', 'jpeg', 'png'))]
        
        for path in self.samples:
            sample = Image.open(path).convert("RGB")  # Cargar imagen
            
            if self.transform:
                sample = self.transform(sample)  # Aplicar la transformación
            
            self.preloaded_data.append((sample, self.label))  # Guardar imagen
            
    def __len__(self):
        return len(self.preloaded_data)

    def __getitem__(self, index):
        return self.preloaded_data[index]

class PreloadedImageFolderTransformations(Dataset):
    def __init__(self, root, label, transform):
        super().__init__()
        self.root = root
        self.transform = transform
        self.label = torch.tensor(label)  # Guardar la etiqueta de la clase
        # Obtener solo archivos de imagen en la carpeta
        self.samples = [os.path.join(root, f) for f in os.listdir(root) if f.lower().endswith(('jpg', 'jpeg', 'png'))]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path = self.samples[index]
        sample = Image.open(path).convert("RGB")  # Cargar imagen
        sample = self.transform(sample)  # Aplicar la transformación
        return (sample, self.label)
    

Create or load the transformed dataset

In [ ]:
def load_dataset(dataroot, dataset_path, image_size, modifiedData):
    if os.path.exists(dataset_path):
        dataset_dir = torch.load(dataset_path)
    else:
        print(f"Creating dataset for {dataset_path}")
        start_time = time.time()
        
        classes = [dir for dir in os.listdir(dataroot) if os.path.isdir(os.path.join(dataroot, dir))]
        dataset_dir = {}
        print(classes)

        for label, dir in enumerate(classes):
            class_path = os.path.join(dataroot, dir)
            print("\tCreating dataset of", class_path)

            if modifiedData:
                train_transforms = transforms.Compose([
                # Recorte aleatorio escalado (manteniendo aspecto) a 224x224
                transforms.Resize(int(image_size * 1.2), interpolation=InterpolationMode.BICUBIC),  # Reescalado
                transforms.RandomResizedCrop(
                    size=image_size, scale=(0.7, 1.0), ratio=(1.0, 1.0),
                    interpolation=InterpolationMode.BICUBIC
                ),
                # Flip horizontal aleatorio
                transforms.RandomHorizontalFlip(p=0.5),
                # AutoAugment con la política de ImageNet
                # transforms.AutoAugment(
                #    policy=AutoAugmentPolicy.IMAGENET,
                #    interpolation=InterpolationMode.BILINEAR
                # ),
                # Sharpennes
                transforms.RandomAdjustSharpness(sharpness_factor=1.5, p=0.5),
                # Variaciones aleatorias de color
                transforms.ColorJitter(
                    brightness=0.2,    # rango más estrecho para brillo
                    contrast=0.2,      # contraste moderado
                    saturation=0.1,    # saturación ligera
                    hue=0.02           # variación mínima en tonos
                ),
                # Conversión a tensor [0,1] 
                transforms.ToTensor(),
                # (Opcional) Borrado aleatorio de un parche
                # transforms.RandomErasing(p=0.1),
                # Normalización con media y desvío estándar de ImageNet
                transforms.Normalize(MEAN, STD)
            ])
            
                # UPDATE IMAGES TRANSFORMATIONS
                dataset_dir[dir] = PreloadedImageFolderTransformations(root=class_path,
                                        label=label,
                                        transform=train_transforms) 
            else:
                # Imagenes originales sin tranformaciones
                dataset_dir[dir] = PreloadedImageFolder(root=class_path,
                                        label=label,
                                        
                                        transform=transforms.Compose([
                                            transforms.Resize(image_size),
                                            transforms.CenterCrop(image_size),
                                            transforms.ToTensor(),
                                            transforms.Normalize(MEAN. STD),
                                        ]))
            

        # show_images_grid(dataset, dataloaderIW_64,"Training Images Imagewoof")
        print("\tSaving at: ", dataset_path)
        torch.save(dataset_dir, dataset_path)
        print(f"\tTotal time creation of the dataset: {time.time() - start_time:.2f} seconds\n")
    return dataset_dir

In [ ]:
def load_all_dataloaders(resolutions, batch_size, path_transformed_dataset, path_dataroot_imagewoof, path_dataroot_imagenette, modifiedData=False, num_workers=0):
    if CREATE_DATALOADERS:
        if modifiedData:
            dataloaders_path = os.path.join(path_transformed_dataset, "dataloadersMod.pt")
            addMod = "_Mod"
        else:
            dataloaders_path = os.path.join(path_transformed_dataset, "dataloaders.pt")
            addMod = ""
            

        if os.path.exists(dataloaders_path):
            dataloader_dicts = torch.load(dataloaders_path)
        else:
            dataloader_dicts = {}
            start_time = time.time()

            for database in DATABASES:
                if database == IMAGENETTE:
                    path_dataroot = path_dataroot_imagenette
                elif database == IMAGEWOOF:
                    path_dataroot = path_dataroot_imagewoof

                for image_size in resolutions:
                    bs = batch_size[image_size]
                    path_dataset = os.path.join(path_transformed_dataset, f"{database}_{image_size}{addMod}.pt")
                    dataset_dir = load_dataset(path_dataroot, path_dataset, image_size, modifiedData)

                    dataloader_key = (database, image_size)
                    if num_workers > 0:
                        dataloader_dicts[dataloader_key] = {
                            dir: DataLoader(dataset_dir[dir], batch_size=bs, shuffle=True,
                                num_workers=num_workers, pin_memory=True, prefetch_factor=6, persistent_workers=True)
                            for dir in dataset_dir.keys()
                        }
                    else:
                        dataloader_dicts[dataloader_key] = {
                            dir: DataLoader(dataset_dir[dir], batch_size=bs, shuffle=True,
                                num_workers=num_workers)
                            for dir in dataset_dir.keys()
                        }
                    
            print("Saving at: ", path_transformed_dataset)
            torch.save(dataloader_dicts, dataloaders_path)
            print(f"Total time creation of all the datasets: {time.time() - start_time:.2f} seconds\n")
    else:
        # Ignore
        dataloader_dicts = {}
        for image_size in resolutions:
            dataloader_dicts[IMAGEWOOF, image_size] = { x: None for x in os.listdir(os.path.join(path_datarootImagewoofTrain))}
            dataloader_dicts[IMAGENETTE, image_size] = { x: None for x in os.listdir(os.path.join(path_datarootImagenetteTrain))}

    return dataloader_dicts


dataloader_dictsMod = load_all_dataloaders(
    resolutions=IMG_SIZES,
    batch_size=BATCH_SIZE,
    path_transformed_dataset=TRANSFORMED_DATASET,
    path_dataroot_imagewoof=path_datarootImagewoofTrain,
    path_dataroot_imagenette=path_datarootImagenetteTrain,
    modifiedData=True,
    num_workers=0
)

dataloader_dicts = load_all_dataloaders(
    resolutions=IMG_SIZES,
    batch_size=BATCH_SIZE,
    path_transformed_dataset=TRANSFORMED_DATASET,
    path_dataroot_imagewoof=path_datarootImagewoofTrain,
    path_dataroot_imagenette=path_datarootImagenetteTrain,
    modifiedData=False,
    num_workers=0
)

Initialize of the weights of the nodes

In [ ]:
# custom weights initialization called on ``netG`` and ``netD``
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

### Models

#### Generator class

##### Generator 64x64

Simple model used with images of size 64x64

In [ ]:
# Generator Code
class Generator_64(nn.Module):
    def __init__(self, ngpu, nz, ngf):
        super(Generator_64, self).__init__()
        self.ngpu = ngpu
        self.nz = nz
        self.ngf = ngf
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. ``(ngf*8) x 4 x 4``
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. ``(ngf*4) x 8 x 8``
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. ``(ngf*2) x 16 x 16``
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # state size. ``(ngf) x 32 x 32``
            nn.ConvTranspose2d(ngf, NC, 4, 2, 1, bias=False),
            nn.Tanh(),
            # state size. ``(NC) x 64 x 64``
        )

    def forward(self, input):
        return self.main(input)

##### Generator 128x128

In [ ]:
# Generator Code

class Generator_128(nn.Module):
    def __init__(self, ngpu, nz, ngf):
        super(Generator_128, self).__init__()
        self.ngpu = ngpu
        self.nz = nz
        self.ngf = ngf
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. ``(ngf*8) x 4 x 4``
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. ``(ngf*4) x 8 x 8``
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. ``(ngf*2) x 16 x 16``
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # state size. ``(ngf) x 32 x 32``
            # NUEVA CAPA
            nn.ConvTranspose2d(ngf, ngf // 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf // 2),
            nn.ReLU(True),
            # state size. ``(ngf // 2) x 64 x 64``
            nn.ConvTranspose2d(ngf // 2, NC, 4, 2, 1, bias=False),
            nn.Tanh(),
            # state size. ``(NC) x 128 x 128``
        )

    def forward(self, input):
        return self.main(input)

##### Generator 256x256

In [ ]:
# Generator Code

class Generator_256(nn.Module):
    def __init__(self, ngpu, nz, ngf):
        super(Generator_256, self).__init__()
        self.ngpu = ngpu
        self.nz = nz
        self.ngf = ngf
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. ``(ngf*8) x 4 x 4``
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. ``(ngf*4) x 8 x 8``
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. ``(ngf*2) x 16 x 16``
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # state size. ``(ngf) x 32 x 32``
            # NUEVAS CAPAS
            nn.ConvTranspose2d(ngf, ngf // 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf // 2),
            nn.ReLU(True),
            # state size. ``(ngf // 2) x 64 x 64``
            nn.ConvTranspose2d(ngf // 2, ngf // 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf // 4),
            nn.ReLU(True),
            # state size. ``(ngf // 4) x 128 x 128``

            nn.ConvTranspose2d(ngf // 4, NC, 4, 2, 1, bias=False),
            nn.Tanh(),
            # state size. ``(NC) x 256 x 256``
        )

    def forward(self, input):
        return self.main(input)

#### Discriminator class

##### Discriminator 64x64

In [ ]:
# Discriminator Code

class Discriminator_64(nn.Module):
    def __init__(self, ngpu, ndf):
        super(Discriminator_64, self).__init__()
        self.ngpu = ngpu
        self.ndf = ndf
        self.main = nn.Sequential(
            # input is ``(NC) x 64 x 64``
            # nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True)
            nn.Conv2d(NC, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf) x 32 x 32``
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*2) x 16 x 16``
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*4) x 8 x 8``
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*8) x 4 x 4``
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, input):
        return self.main(input)

##### Discriminator 128x128

In [ ]:
# Discriminator Code 
class Discriminator_128(nn.Module):
    def __init__(self, ngpu, ndf):
        super(Discriminator_128, self).__init__()
        self.ngpu = ngpu
        self.ndf = ndf
        self.main = nn.Sequential(
            # NUEVA CAPA: input (NC) x 128 x 128 -> (ndf//2) x 64 x 64
            nn.Conv2d(NC, ndf // 2, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # input is ``(NC) x 64 x 64``
            # nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True)
            nn.Conv2d(ndf // 2, ndf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf) x 32 x 32``
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*2) x 16 x 16``
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*4) x 8 x 8``
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*8) x 4 x 4``
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, input):
        return self.main(input)

##### Discriminator 256x256

In [ ]:
# Discriminator Code 
class Discriminator_256(nn.Module):
    def __init__(self, ngpu, ndf):
        super(Discriminator_256, self).__init__()
        self.ngpu = ngpu
        self.ndf = ndf
        self.main = nn.Sequential(
            # NUEVA CAPA: input (NC) x 128 x 128 -> (ndf//2) x 64 x 64
            nn.Conv2d(NC, ndf // 4, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # input is ``(NC) x 64 x 64``
            # nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True)
            nn.Conv2d(ndf // 4, ndf // 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf // 2),
            nn.LeakyReLU(0.2, inplace=True),
            # nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True)
            nn.Conv2d(ndf // 2, ndf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf) x 32 x 32``
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*2) x 16 x 16``
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*4) x 8 x 8``
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*8) x 4 x 4``
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, input):
        return self.main(input)

#### Create model

In [ ]:
def create_model(model_class, device, *args, verbose=False):
    if CREATE_MODELS:
        # Extraemos ngpu como el primer elemento de args
        ngpu, *rest_args = args

        # Instanciamos el modelo con los argumentos restantes
        model = model_class(ngpu, *rest_args)

        if ((device.type == 'cuda') and (ngpu > 1)):
            model = nn.DataParallel(model, list(range(ngpu)))

        model.apply(weights_init)

        if verbose:
            print(model)

        return model

In [ ]:
fixed_noise = {}
for nz in NZ_VALUES:
    fixed_noise[nz] = torch.randn(NUM_IMAGES_FOR_STEP, nz, 1, 1, device=device)


def create_new_model(num, model_list, netD, netG, lr, dataloader, dataloaderName, num_epochs, className, image_size, nz, discriminatorUpdate=1, generatorUpdate=1, discriminatorLrMultiply=1, generatorLrMultiply=1, betas=None, step_size=None, gama=None, modifiedDataset=True):
    if CREATE_MODELS:
        netD_copy = copy.deepcopy(netD)
        netG_copy = copy.deepcopy(netG)

        if betas is None:
            betas = (beta1, 0.999)

        optimizerD = optim.Adam(netD_copy.parameters(), lr=lr * discriminatorLrMultiply, betas=(beta1, 0.999))
        optimizerG = optim.Adam(netG_copy.parameters(), lr=lr * generatorLrMultiply, betas=(beta1, 0.999))

        if (step_size is None and gama is None):
            schedulerD = None
            schedulerG = None
        else:
            schedulerD = torch.optim.lr_scheduler.StepLR(optimizerD, step_size=step_size, gamma=gama)
            schedulerG = torch.optim.lr_scheduler.StepLR(optimizerG, step_size=step_size, gamma=gama)
        
        if num_epochs > 0:
            model_list.append([num, netD_copy, optimizerD, schedulerD, netG_copy, optimizerG, schedulerG, dataloader, dataloaderName, num_epochs, className, lr, fixed_noise[nz], image_size, nz, discriminatorUpdate, generatorUpdate, discriminatorLrMultiply, generatorLrMultiply, modifiedDataset])


#### Optimizer

In [ ]:
# Initialize the ``BCELoss`` function
criterion = nn.BCELoss()

# Establish convention for real and fake labels during training
real_label = 1.
fake_label = 0.


### Different Models to train

##### Discriminator Initialization

In [ ]:
netD_64  = create_model(Discriminator_64, device, NGPU, 64, verbose=False)
netD_128  = create_model(Discriminator_128, device, NGPU, 128, verbose=False)
netD_256  = create_model(Discriminator_256, device, NGPU, 256, verbose=False)

discriminators_dict = {
    64: netD_64,
    128: netD_128,
    256: netD_256
}

##### Generator Initialization

In [ ]:
generators_dict = {}
for image_size in IMG_SIZES:
    generators_dict[image_size] = {}        


generators_64 = []
image_size = 64
if image_size in IMG_SIZES:
    for nz in NZ_VALUES:
        netG = create_model(Generator_64, device, NGPU, nz, image_size, verbose=False)
        generators_64.append((netG, nz))
        generators_dict[image_size][nz] = netG

generators_128 = []
image_size = 128
if image_size in IMG_SIZES:
    for nz in NZ_VALUES:
        netG = create_model(Generator_128, device, NGPU, nz, image_size, verbose=False)
        generators_128.append((netG, nz))
        generators_dict[image_size][nz] = netG

generators_256 = []
image_size = 256
if image_size in IMG_SIZES:
    for nz in NZ_VALUES:
        netG = create_model(Generator_256, device, NGPU, nz, image_size, verbose=False)
        generators_256.append((netG, nz))
        generators_dict[image_size][nz] = netG


### Train Models

In [ ]:
# Training Loop

def save_movel(filename, model, optimizer, scheduler, epoch, losses, accuracy, img_list=None, num_images=None, trained_time=None):
    if scheduler is not None:
        scheduler = scheduler.state_dict()

    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer': optimizer,
        'scheduler_state_dict': scheduler,
        'epoch': epoch,
        'losses': losses,
        'accuracy': accuracy,
        'img_list': img_list,
        'num_images': num_images,
        'trainedTime': trained_time,
    }, filename)


def train(num, modelD_filename, netD, optimizerD, schedulerD, modelG_filename, netG, optimizerG, schedulerG, dataloader, trained_epochs, num_epochs, 
          image_size, nz, fixed_noise, discriminatorUpdate, generatorUpdate, D_accuracy=None, G_accuracy=None, D_losses=None, G_losses=None, img_list=None, num_images=None, trained_time=None):

    # Lists to keep track of progress
    if img_list is None:
        # Crear un tensor vacío con forma (0, 3, 64, 64)
        img_list = torch.empty((0, 3, image_size, image_size))
    if num_images is None:
        num_images = 0

    def expand_list(lst, num_epochs, update=1):
        if lst is None:
            return [None] * (num_epochs // update)
        else:
            if len(lst) < (num_epochs // update):
                lst.extend([None] * ((num_epochs // update) - len(lst)))
            return lst

    trained_time = expand_list(trained_time, num_epochs, generatorUpdate)
    D_accuracy = expand_list(D_accuracy, num_epochs, discriminatorUpdate)
    G_accuracy = expand_list(G_accuracy, num_epochs, generatorUpdate)
    D_losses = expand_list(D_losses, num_epochs, discriminatorUpdate)
    G_losses = expand_list(G_losses, num_epochs, generatorUpdate)

    last_errD = 0.0
    last_D_x = 0.0
    last_D_G_z1 = 0.0

    netG = netG.to(device)
    netD = netD.to(device)
    
    # For each epoch
    for i, epoch in enumerate(range(trained_epochs, num_epochs)):
        # For each batch in the dataloader
        start_time = time.time()
        for data in dataloader:
            ############################
            # (1) Update D network: maximize log(D(x)) + log(1 - D(G(z)))
            ###########################
            if ((epoch % discriminatorUpdate) == 0 or (epoch == trained_epochs)):
                ## Train with all-real batch
                netD.zero_grad()
                # Format batch
                real_cpu = data[0].to(device)
                b_size = real_cpu.size(0)
                label = torch.full((b_size,), real_label, dtype=torch.float, device=device)
                # Forward pass real batch through D
                output = netD(real_cpu).view(-1)
                # Calculate loss on all-real batch
                errD_real = criterion(output, label)
                # Calculate gradients for D in backward pass
                errD_real.backward()
                D_x = output.mean().item()

                ## Train with all-fake batch
                # Generate batch of latent vectors
                noise = torch.randn(b_size, nz, 1, 1, device=device)
                # Generate fake image batch with G
                fake = netG(noise)
                label.fill_(fake_label)
                # Classify all fake batch with D
                output = netD(fake.detach()).view(-1)
                # Calculate D's loss on the all-fake batch
                errD_fake = criterion(output, label)
                # Calculate the gradients for this batch, accumulated (summed) with previous gradients
                errD_fake.backward()
                D_G_z1 = output.mean().item()
                # Compute error of D as sum over the fake and the real batches
                errD = errD_real + errD_fake
                # Update D
                optimizerD.step()
                # Guardar los valores actuales
                last_errD = errD.item()
                last_D_x = D_x
                last_D_G_z1 = D_G_z1
                if schedulerD is not None:    
                    schedulerD.step()  # Update the learning rate based on the scheduler
            else:
                noise = torch.randn(b_size, nz, 1, 1, device=device)
                # Generate fake image batch with G
                fake = netG(noise)
                label.fill_(fake_label)
                # Recuperar valores anteriores si no se actualiza D
                errD = torch.tensor(last_errD)
                D_x = last_D_x
                D_G_z1 = last_D_G_z1

            ############################
            # (2) Update G network: maximize log(D(G(z)))
            ###########################
            if ((epoch % generatorUpdate) == 0 or (epoch == trained_epochs)):
                netG.zero_grad()
                label.fill_(real_label)  # fake labels are real for generator cost
                # Since we just updated D, perform another forward pass of all-fake batch through D
                output = netD(fake).view(-1)
                # Calculate G's loss based on this output
                errG = criterion(output, label)
                # Calculate gradients for G
                errG.backward()
                D_G_z2 = output.mean().item()
                # Update G
                optimizerG.step()
                if schedulerG is not None:
                    schedulerG.step()  # Update the learning rate based on the scheduler
            else:
                # No se actualiza G: se usan los valores anteriores
                errG = torch.tensor(G_losses[-1]) if G_losses else torch.tensor(0.0)
                D_G_z2 = D_G_z2 if 'D_G_z2' in locals() else 0.0

            # Save Losses for plotting later
            G_losses[i//generatorUpdate] = errG.item()
            G_accuracy[i//generatorUpdate] = D_G_z1

            D_losses[i//discriminatorUpdate] = errD.item()
            D_accuracy[i//discriminatorUpdate] = D_x
        # END OF EPOCH
        if (torch.cuda.is_available() and ((torch.cuda.memory_allocated() / 1024**2 > 10000) or (torch.cuda.memory_reserved() / 1024**2 > 10000))): # Check if the gpu is too full, reduce batch size
            print("************************************************************ WARNING ************************************************************")
            print(f"CUDA memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
            print(f"CUDA memory reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")
        # Output training stats
        if schedulerG is not None:
            schedulerG.step()
        
        # Save the model state
        if (((epoch+1) % ROUNDS_PER_SAVE_IMAGES) == 0 or ((epoch+1) == num_epochs)):
            print('[%d/%d]\tLoss_D: %.4f\tLoss_G: %.4f\tD(x): %.4f\tD(G(z)): %.4f / %.4f' % (epoch+1, num_epochs, errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))
            with torch.no_grad():
                fake = netG(fixed_noise).detach().cpu()
                # Normalizar imágenes de [-1,1] a [0,1]
                fake = (fake + 1) / 2
                img_list = torch.cat((fake, img_list), dim=0)  # Concat the new images
                num_images += 1
                
        if (((epoch+1) % ROUNDS_PER_SAVE_MODELS) == 0 or ((epoch+1) == num_epochs)):
            # Save the model state
            save_movel(modelD_filename, netD, optimizerD, schedulerD, epoch+1, D_losses, D_accuracy)
            save_movel(modelG_filename, netG, optimizerG, schedulerG, epoch+1, G_losses, G_accuracy, img_list, num_images, trained_time)

        trained_time[epoch - 1] = (time.time() - start_time)
    
    # END OF TRAINING
    print(f"\tModel {num} trained for {num_epochs} epochs. Time spent: {sum(t for t in trained_time if t is not None):.2f} seconds")


def load_all_models(models, verbose=True, trained_models=None):
    if trained_models is None:
        trained_models = {}
    total_models = len(models)
    for i, model in enumerate(models, 1):
        num, netD, optimizerD, schedulerD, netG, optimizerG, schedulerG, dataloader, dataloaderName, num_epochs, className, lr, fixed_noise, image_size, nz, discriminatorUpdate, generatorUpdate, discriminatorLrMultiply, generatorLrMultiply, modifiedDataset = model
        modifiedDataset = 1 if modifiedDataset == True else 0
        if image_size in IMG_SIZES: # Solo para debuguear
            lrD = "{:.6f}".format(lr*discriminatorLrMultiply)
            lrG = "{:.6f}".format(lr*generatorLrMultiply)
            modelD_filename = os.path.join(MODELS, dataloaderName, f"nz={nz}_lrD={lrD}_lrG={lrG}_imgSize={image_size}_uD={discriminatorUpdate}_uG={generatorUpdate}_modData={modifiedDataset}_ModelD-{className}.pt")
            modelG_filename = os.path.join(MODELS, dataloaderName, f"nz={nz}_lrD={lrD}_lrG={lrG}_imgSize={image_size}_uD={discriminatorUpdate}_uG={generatorUpdate}_modData={modifiedDataset}_ModelG-{className}.pt")
            print(modelD_filename, modelG_filename)
            if TRAIN_MODELS:
                if os.path.isfile(modelD_filename) and os.path.isfile(modelG_filename):
                    # Cambiar el como se carga porque no se estan entrenando una vez fueron generados

                    # Discriminator
                    checkpoint = torch.load(modelD_filename)
                    netD.load_state_dict(checkpoint['model_state_dict'])
                    optimizerD = checkpoint['optimizer']
                    if checkpoint['scheduler_state_dict'] is not None:
                        schedulerD.load_state_dict(checkpoint['scheduler_state_dict'])
                    else:
                        schedulerD = None
                    trained_epochs = checkpoint['epoch']
                    D_losses = checkpoint['losses']
                    D_accuracy = checkpoint['accuracy']

                    # Genetator
                    checkpoint = torch.load(modelG_filename)
                    netG.load_state_dict(checkpoint['model_state_dict'])
                    optimizerG = checkpoint['optimizer']
                    if checkpoint['scheduler_state_dict'] is not None:
                        schedulerG.load_state_dict(checkpoint['scheduler_state_dict'])
                    else:
                        schedulerG = None
                    
                    img_list = checkpoint['img_list']
                    G_losses = checkpoint['losses']
                    G_accuracy = checkpoint['accuracy']
                    trained_time = checkpoint['trainedTime']
                    num_images = checkpoint['num_images']

                    if verbose:
                        print(f"Model {num}.({i}/{total_models}), loaded from saved file. {modelD_filename.replace(MODELS + os.sep, '')} - {modelG_filename.replace(MODELS + os.sep, '')}")

                    if trained_epochs < num_epochs:
                        if verbose:
                            print(f"\tModel trained for {trained_epochs}/{num_epochs}. Time spent: {sum(t for t in trained_time if t is not None):.2f} seconds")
                        train(num, modelD_filename, netD, optimizerD, schedulerD, modelG_filename, netG, optimizerG, schedulerG, dataloader, 
                            trained_epochs, num_epochs, image_size, nz, fixed_noise, discriminatorUpdate, generatorUpdate, D_accuracy, G_accuracy, D_losses, G_losses, img_list, num_images, trained_time)
                    else:
                        if verbose:
                            print(f"\tModel trained for {trained_epochs}/{trained_epochs}. Time spent: {sum(t for t in trained_time if t is not None):.2f} seconds")
                else:
                    if verbose:
                        print(f"Training model {num}.({i}/{total_models}), {modelD_filename.replace(MODELS + os.sep, '')} - {modelG_filename.replace(MODELS + os.sep, '')}")
                
                    train(num, modelD_filename, netD, optimizerD, schedulerD, modelG_filename, netG, optimizerG, schedulerG, dataloader, 
                        0, num_epochs, image_size, nz, fixed_noise, discriminatorUpdate, generatorUpdate)
            
            if num not in trained_models:
                trained_models[num] = []
            if [modelD_filename, modelG_filename, dataloaderName] not in trained_models[num]:
                trained_models[num].append([modelD_filename, modelG_filename, dataloaderName])

            for idx in [1, 2, 3, 4, 5, 6, 13]:
                model[idx] = None
    
    del models
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    gc.collect()
    # print(f"[GPU] Used: {torch.cuda.memory_allocated() / 1024**2:.2f} MB | Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")
    return trained_models


In [ ]:
def listar_tensores_en_cuda():
    print("🔍 Tensores actualmente en GPU:\n")
    for obj in gc.get_objects():
        try:
            if (torch.is_tensor(obj) and obj.is_cuda):
                print(f"Tipo: {type(obj)}, Shape: {obj.size()}, Device: {obj.device}")
        except Exception:
            pass

In [ ]:
def fillModelListFromImageSize(image_size, database, dataloaderDict, modifiedDataset, discriminatorUpdate=1, generatorUpdate=1, discriminatorLrMultiply=1, generatorLrMultiply=1, num=1):
    modelsCreated = [] # Append the differrent models of the Discriminator and Generator and their parameters
    if image_size in IMG_SIZES:
        netD = discriminators_dict[image_size]
        dataloaderName = f"{database}_64"
        classifier = classifierFromName[database]
        dataloader = dataloaderDict[(database, image_size)]

        # One model for each image class
        for nz in NZ_VALUES:
            for lr in LEARNING_RATES:
                for  dir in classifier.values():
                    dataloader_class = dataloader[dir]
                    create_new_model(num, modelsCreated, netD, generators_dict[image_size][nz], lr, dataloader_class, dataloaderName, NUM_EPOCHS, dir, image_size, nz, 
                                    discriminatorUpdate=discriminatorUpdate, generatorUpdate=generatorUpdate, discriminatorLrMultiply=discriminatorLrMultiply, 
                                    generatorLrMultiply=generatorLrMultiply, modifiedDataset=modifiedDataset)
                num += 1
                
    return modelsCreated

In [ ]:
trained_models = {}

In [ ]:
def append_dicts(dict1, dict2):
    result = defaultdict(list)

    for d in (dict1, dict2):
        for key, value in d.items():
            result[key].append(value)

    return dict(result)

#### Models 64x64

##### Train Imagenette models 64x64

In [ ]:
trained_models[(64, IMAGENETTE)] = load_all_models(fillModelListFromImageSize(64, IMAGENETTE, dataloader_dicts, modifiedDataset=False))

In [ ]:
trained_models[(64, IMAGENETTE)] = load_all_models(fillModelListFromImageSize(64, IMAGENETTE, dataloader_dictsMod, modifiedDataset=True, discriminatorLrMultiply=1, generatorLrMultiply=1.3, 
                                                                              num=(len(trained_models[(64, IMAGENETTE)])+1)),
                                                                              trained_models=trained_models[(64, IMAGENETTE)]
                                                                              )

##### Train Imagewoof models 64x64

In [ ]:
trained_models[(64, IMAGEWOOF)] = load_all_models(fillModelListFromImageSize(64, IMAGEWOOF, dataloader_dicts, modifiedDataset=False))

In [ ]:
trained_models[(64, IMAGEWOOF)] = load_all_models(fillModelListFromImageSize(64, IMAGEWOOF, dataloader_dictsMod, modifiedDataset=True, discriminatorLrMultiply=1, generatorLrMultiply=1.3, 
                                                                              num=(len(trained_models[(64, IMAGEWOOF)])+1)),
                                                                              trained_models=trained_models[(64, IMAGEWOOF)]
                                                                              )


#### Models 128x128

##### Train Imagenette models 128x128

In [ ]:
trained_models[(128, IMAGENETTE)] = load_all_models(fillModelListFromImageSize(128, IMAGENETTE, dataloader_dicts, modifiedDataset=False))

In [ ]:
trained_models[(128, IMAGENETTE)] = load_all_models(fillModelListFromImageSize(128, IMAGENETTE, dataloader_dictsMod, modifiedDataset=True, discriminatorLrMultiply=1, generatorLrMultiply=1.3, 
                                                                              num=(len(trained_models[(128, IMAGENETTE)])+1)))


##### Train Imagewoof models 128x128

In [ ]:
trained_models[(128, IMAGEWOOF)] = load_all_models(fillModelListFromImageSize(128, IMAGEWOOF, dataloader_dicts, modifiedDataset=False))

In [ ]:
trained_models[(128, IMAGEWOOF)] = load_all_models(fillModelListFromImageSize(128, IMAGEWOOF, dataloader_dictsMod, modifiedDataset=True, discriminatorLrMultiply=1, generatorLrMultiply=1.3, 
                                                                              num=(len(trained_models[(128, IMAGEWOOF)])+1)))


#### Models 256x256

##### Train Imagenette models 256x256

In [ ]:
modelsIW_256 = [] # Append the differrent models of the Discriminator and Generator. Each append consist of (Discrimnator, OptimizerD, Generator, OptimizerG)
modelsIN_256 = []
image_size = 256

# One model for each image class
if image_size in IMG_SIZES:
    netD = netD_256

    # IMAGEWOOF
    dataloaderName = f"{IMAGEWOOF}_256"
    classifier = classifierImagewoof
    dataloader = dataloader_dicts[(IMAGEWOOF, 256)]
    num = 1
    for netG, nz in generators_256:
        for lr in LEARNING_RATES:
            for  dir in classifier.values():
                dataloader_class = dataloader[dir]
                create_new_model(num, modelsIW_256, netD, netG, lr, dataloader_class, dataloaderName, NUM_EPOCHS, dir, image_size, nz, 
                                discriminatorUpdate=1, generatorUpdate=1, discriminatorLrMultiply=0.25, generatorLrMultiply=2)
            num += 1
    
    # IMAGENETTE
    dataloaderName = f"{IMAGENETTE}_256"
    classifier = classifierImagenette
    dataloader = dataloader_dicts[(IMAGENETTE, 256)]

    # One model for each image class
    num = 1
    for netG, nz in generators_256:
        for lr in LEARNING_RATES:
            for dir in classifierImagenette.values():
                dataloader_class = dataloader[dir]
                create_new_model(num, modelsIN_256, netD, netG, lr, dataloader_class, dataloaderName, NUM_EPOCHS, dir, image_size, nz, 
                                discriminatorUpdate=1, generatorUpdate=1, discriminatorLrMultiply=0.25, generatorLrMultiply=2)
            num += 1

In [ ]:
trained_models[(256, IMAGENETTE)] = load_all_models(modelsIN_256)

##### Train Imagewoof models 256x256

In [ ]:
trained_models[(256, IMAGEWOOF)] = load_all_models(modelsIW_256)

### Results

#### Loss of the models

Code of the function we will use to plot

In [ ]:
def plot_loss(models):
    if PLOT_LOSS:
        for num, model in enumerate(models.keys(), 1):
            
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
            
            for pair_model in models[model]: 
                modelD_filename, modelG_filename, datasetTitle = pair_model

                if IMAGENETTE in datasetTitle:
                    classifier = classifierImagenette_inv
                    datasetName = IMAGENETTE
                elif IMAGEWOOF in datasetTitle:
                    classifier = classifierImagewoof_inv
                    datasetName = IMAGEWOOF
                else:
                    print("Invalid dataset")
                    break

                if os.path.isfile(modelD_filename) and os.path.isfile(modelG_filename):
                    checkpoint = torch.load(modelD_filename)
                    D_losses = checkpoint['losses']

                    checkpoint = torch.load(modelG_filename)
                    G_losses = checkpoint['losses']

                    pattern = r"nz=(\d+)_lrD=([\d.]+)_lrG=([\d.]+)_imgSize=(\d+)_uD=(\d+)_uG=(\d+)_modData=(\d+)_ModelD-([a-z0-9]+)"

                    match = re.match(pattern, os.path.basename(modelD_filename))
                    if match:
                        nz = int(match.group(1))
                        lrD = float(match.group(2))
                        lrG = float(match.group(3))
                        imgSize = int(match.group(4))
                        uD = int(match.group(5))
                        uG = int(match.group(6))
                        modData = int(match.group(7))
                        n = match.group(8)
                    else:
                        print("La cadena no coincide con el patrón.")

                    ax1.plot(G_losses, label=f"G: {classifier[n]}")
                    ax2.plot(D_losses, label=f"D: {classifier[n]}")

            fig.suptitle(f"{num}. {datasetName} - Loss During Training (nz={nz}, lrD={lrD}, lrG={lrG}, imgSize={imgSize}, uD={uD}, uG={uG}), modData={modData}", fontsize=16)

            ax1.set_title("Generator Loss During Training")
            ax1.set_ylabel("Generator Loss")
            ax2.set_xlabel("Iterations")
            ax1.legend()

            ax2.set_title("Discriminator Loss During Training")
            ax2.set_xlabel("Iterations")
            ax2.set_ylabel("Discriminator Loss")
            ax2.legend()

            plt.tight_layout()
            plt.show()

##### Imagenette 64x64:

In [ ]:
plot_loss(trained_models[64, IMAGENETTE])

##### Imagewoof 64x64

In [ ]:
plot_loss(trained_models[64, IMAGEWOOF])

##### Imagenette 128x128

In [ ]:
plot_loss(trained_models[128, IMAGENETTE])

##### Imagewoof 128x128

In [ ]:
plot_loss(trained_models[128, IMAGEWOOF])

##### Imagenette 256x256

In [ ]:
plot_loss(trained_models[256, IMAGENETTE])

##### Imagewoof 256x256

In [ ]:
plot_loss(trained_models[256, IMAGEWOOF])

#### Models Learning Progress

##### Accuracy evolution

In [ ]:
def plot_accuracy_evolution(models):
    if PLOT_ACCURACY_EVOLUTION:
        for num, model in enumerate(models.keys(), 1):
            
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
            
            for pair_model in models[model]: 
                modelD_filename, modelG_filename, datasetTitle = pair_model

                if IMAGENETTE in datasetTitle:
                    classifier = classifierImagenette_inv
                    datasetName = IMAGENETTE
                elif IMAGEWOOF in datasetTitle:
                    classifier = classifierImagewoof_inv
                    datasetName = IMAGEWOOF
                else:
                    print("Invalid dataset")
                    break

                if os.path.isfile(modelD_filename) and os.path.isfile(modelG_filename):
                    checkpoint = torch.load(modelD_filename)
                    D_losses = checkpoint['accuracy']

                    checkpoint = torch.load(modelG_filename)
                    G_losses = checkpoint['accuracy']

                    pattern = r"nz=(\d+)_lrD=([\d.]+)_lrG=([\d.]+)_imgSize=(\d+)_uD=(\d+)_uG=(\d+)_modData=(\d+)_ModelD-([a-z0-9]+)"

                    match = re.match(pattern, os.path.basename(modelD_filename))
                    if match:
                        nz = int(match.group(1))
                        lrD = float(match.group(2))
                        lrG = float(match.group(3))
                        imgSize = int(match.group(4))
                        uD = int(match.group(5))
                        uG = int(match.group(6))
                        modData = int(match.group(7))
                        n = match.group(8)
                    else:
                        print("La cadena no coincide con el patrón.")

                    ax1.plot(G_losses, label=f"G: {classifier[n]}")
                    ax2.plot(D_losses, label=f"D: {classifier[n]}")

            fig.suptitle(f"{num}. {datasetName} - Accuracy Evolution (nz={nz}, lrD={lrD}, lrG={lrG}, imgSize={imgSize}, uD={uD}, uG={uG}), modData={modData}", fontsize=16)
            plt.ylim(0, 1)

            ax1.set_title("Generator Accuracy During Training (% of fake images classified as real)")
            ax1.set_ylabel("Generator Accuracy")
            ax2.set_xlabel("Iterations")
            ax1.legend()

            ax2.set_title("Discriminator Accuracy During Training (% of real images classified as real)")
            ax2.set_xlabel("Iterations")
            ax2.set_ylabel("Discriminator Accuracy")
            ax2.legend()

            plt.tight_layout()
            plt.show()

###### Imagenette 64x64:

In [ ]:
plot_accuracy_evolution(trained_models[64, IMAGENETTE])

###### Imagewoof 64x64

In [ ]:
plot_accuracy_evolution(trained_models[64, IMAGEWOOF])

###### Imagenette 128x128

In [ ]:
plot_accuracy_evolution(trained_models[128, IMAGENETTE])

###### Imagewoof 128x128

In [ ]:
plot_accuracy_evolution(trained_models[128, IMAGEWOOF])

###### Imagenette 256x256

In [ ]:
plot_accuracy_evolution(trained_models[256, IMAGENETTE])

###### Imagewoof 256x256

In [ ]:
plot_accuracy_evolution(trained_models[256, IMAGEWOOF])

##### Image creation progress 

Code of the function we will use to visualize the results

In [ ]:
INCREMENT = 75

In [ ]:
def show_image_results(models, img_size, increment=None):
    if SHOW_PROGRESS_IMAGES:
        for num, model in enumerate(models.keys(), 1):
            # Crear un tensor vacío con forma (0, 3, 64, 64)
            total_models = len(models[model])

            print("\n\nShowing Results of Model", num)
            for i in range(total_models):
                _, modelG_filename, datasetTitle = models[model][i]

                if os.path.isfile(modelG_filename):
                    checkpoint = torch.load(modelG_filename)
                    img_list = torch.empty((0, 3, img_size, img_size))
                    model_images = checkpoint['img_list']
                    #num_images = checkpoint['num_images']
                    epochs = checkpoint['epoch']
                    # Only add images if num_images % increment == 0

                    def get_values_to_plot(N, M, X):
                        """
                        N: número total de iteraciones (ej. 500)
                        M: cada cuántas iteraciones se guardó un valor (ej. 50)
                        X: cada cuántas iteraciones quiero mostrar un resultado (ej. 100)

                        Devuelve una lista de índices que se pueden usar para acceder 
                        a los elementos correctos en un array que guarda datos cada M iteraciones,
                        para así mostrar el valor correspondiente a cada X iteraciones reales.
                        """
                        return [i // M for i in range(0, N + 1, X) if i % M == 0]
                    
                    indices_to_plot = get_values_to_plot(epochs, ROUNDS_PER_SAVE_IMAGES, increment)

                    for i in indices_to_plot:
                        # img_list = torch.cat((img_list, model_images[i].unsqueeze(0)), dim=0)
                        fromIndex = i*NUM_IMAGES_FOR_STEP
                        toIndex = (i+1)*NUM_IMAGES_FOR_STEP
                        if fromIndex > NUM_IMAGES_FOR_STEP:
                            fromIndex -= NUM_IMAGES_FOR_STEP
                            toIndex -= NUM_IMAGES_FOR_STEP
                        img_list = torch.cat((img_list, model_images[fromIndex: toIndex]), dim=0)
                    # img_list = torch.cat((img_list, model_images), dim=0)
                    grid_img = vutils.make_grid(img_list, nrow=NUM_IMAGES_FOR_STEP, normalize=True)
                    np_img = grid_img.permute(1, 2, 0).numpy()

                    # Usa fig + ax para mantener control total sobre la figura
                    fig, ax = plt.subplots(figsize=(20, 20))
                    ax.imshow(np_img)
                    ax.axis('off')
                    if IMAGENETTE in datasetTitle:
                        classifier = classifierImagenette_inv
                        datasetName = IMAGENETTE
                    elif IMAGEWOOF in datasetTitle:
                        classifier = classifierImagewoof_inv
                        datasetName = IMAGEWOOF
                    else:
                        print("Invalid dataset")
                        break

                    if os.path.isfile(modelG_filename):
                        checkpoint = torch.load(modelG_filename)

                        pattern = r"nz=(\d+)_lrD=([\d.]+)_lrG=([\d.]+)_imgSize=(\d+)_uD=(\d+)_uG=(\d+)_modData=(\d+)_ModelG-([a-z0-9]+)"

                        match = re.match(pattern, os.path.basename(modelG_filename))

                        if match:
                            nz = int(match.group(1))
                            lrD = float(match.group(2))
                            lrG = float(match.group(3))
                            imgSize = int(match.group(4))
                            uD = int(match.group(5))
                            uG = int(match.group(6))
                            modData = int(match.group(7))
                            n = match.group(8)
                        else:
                            print(os.path.basename(modelG_filename))
                            print("La cadena no coincide con el patrón.")
                    else:
                        print("Model not found")
        
                    # Añadir título a la figura
                    ax.set_title(rf"{num}.{datasetName} Generator evolution during training for $\bf{{{classifier[n]}}}$. nz={nz}, lrD={lrD}, lrG={lrG}, uG={uG}, uD={uD}, imgSize={imgSize}, modData={modData}", fontsize=16)

                    # Añadir texto por fila (una etiqueta por iteración)
                    numbers = list(indices_to_plot)[::-1]
                    num_rows = len(list(indices_to_plot))
                    img_height = img_size  # o 128 si estás generando imágenes más grandes

                    for i in range(num_rows):
                        y = i * img_height + img_height // 2
                        ax.text(-10, y, f"Epoch {numbers[i]*ROUNDS_PER_SAVE_IMAGES}", va='center', ha='right',
                                fontsize=12, color='white', backgroundcolor='black')

                    plt.tight_layout()
                    plt.show()

###### Imagenette 64x64

In [ ]:
show_image_results(trained_models[(64, IMAGENETTE)], 64, INCREMENT)

###### Imagewoof 64x64

In [ ]:
show_image_results(trained_models[(64, IMAGEWOOF)], 64, INCREMENT)

###### Imagenette 128x128

In [ ]:
show_image_results(trained_models[(128, IMAGENETTE)], 128, INCREMENT)

###### Imagewoof 128x128

In [ ]:
show_image_results(trained_models[(128, IMAGEWOOF)], 128, INCREMENT)

###### Imagenette 256x256

In [ ]:
show_image_results(trained_models[(256, IMAGENETTE)], 256, INCREMENT)

###### Imagewoof 256x256

In [ ]:
show_image_results(trained_models[(256, IMAGEWOOF)], 256, INCREMENT)

#### Comparison

Compare the real photos to the fake ones of each model

In [ ]:
def compare_results(models, dataloaderName, image_size):
    # Selección del dataloader
    if SHOW_FINAL_RESULTS:
        if dataloaderName == IMAGENETTE:
            classifier = classifierImagenette_inv
            datasetName = IMAGENETTE
            dataroot = datarootImagenette
        elif dataloaderName == IMAGEWOOF:
            classifier = classifierImagewoof_inv
            datasetName = IMAGEWOOF
            dataroot = datarootImagewoof
        else:
            print("Not recognized dataloader")
            return

        def load_real_images(folder_path, max_images=8):
            """Carga hasta max_images imágenes reales redimensionadas como tensores"""
            images = []
            count = 0
            resize_transform = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor()
            ])
            for file in sorted(os.listdir(folder_path)):
                if file.lower().endswith(('.jpeg', '.jpg', '.png')):
                    path = os.path.join(folder_path, file)
                    try:
                        img = Image.open(path).convert('RGB')
                        img_tensor = resize_transform(img)
                        images.append(img_tensor)
                        count += 1
                        if count == max_images:
                            break
                    except Exception as e:
                        print(f"Error abriendo imagen: {path}. Error: {e}")
            return images

        for num, model_name in enumerate(models.keys(), 1):
            fig = plt.figure(figsize=(15, 15))

            generated_images = []
            real_images = []
            labels = []

            for _, modelG_filename, _ in models[model_name]:
                # Cargar checkpoint del modelo
                if not os.path.isfile(modelG_filename):
                    continue

                checkpoint = torch.load(modelG_filename)
                model_images = checkpoint['img_list'][:8]
                generated_images.append(model_images)

                pattern = r"nz=(\d+)_lrD=([\d.]+)_lrG=([\d.]+)_imgSize=(\d+)_uD=(\d+)_uG=(\d+)_modData=(\d+)_ModelG-([a-z0-9]+)"

                match = re.match(pattern, os.path.basename(modelG_filename))
                if match:
                    nz = int(match.group(1))
                    lrD = float(match.group(2))
                    lrG = float(match.group(3))
                    imgSize = int(match.group(4))
                    uD = int(match.group(5))
                    uG = int(match.group(6))
                    modData = int(match.group(7))
                    n = match.group(8)
                else:
                    print("La cadena no coincide con el patrón.")

                labels.append(classifier[n])

                # Cargar imágenes reales
                real_folder = os.path.join(dataroot, "train", n)
                real_images += load_real_images(real_folder)

            # Concatenar imágenes generadas
            if (len(generated_images) == 0 or len(real_images) == 0):
                print(f"Saltando {model_name} por falta de imágenes.")
                continue

            img_list = torch.cat(generated_images, dim=0)
            img_list_real = torch.stack(real_images)

            # Mostrar Figura 1: Imágenes generadas
            plt.subplot(1, 2, 1)
            plt.axis("off")
            plt.title(f"{num}.{datasetName} Generator Results\nnz={nz}, lrG={lrG}, lrD={lrD}, uG={uG}, uD={uD}, imgSize={imgSize}, modData={modData}")

            grid = vutils.make_grid(img_list, nrow=8, normalize=True)
            grid_np = grid.permute(1, 2, 0).cpu().numpy()
            plt.imshow(grid_np)

            # Añadir etiquetas por fila
            alto_total = grid_np.shape[0]
            alto_fila = alto_total / len(labels)
            for i, label in enumerate(labels):
                plt.text(-10, i * alto_fila + alto_fila / 2, label,
                        va='center', ha='right', fontsize=10,
                        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

            # Mostrar Figura 2: Imágenes reales
            plt.subplot(1, 2, 2)
            plt.axis("off")
            plt.title(f"{num}. Real Images")

            grid_real = vutils.make_grid(img_list_real, nrow=8, padding=5, normalize=True)
            grid_real_np = np.transpose(grid_real.cpu().numpy(), (1, 2, 0))
            plt.imshow(grid_real_np)

            plt.tight_layout()
            plt.show()

##### Imagenette Comparison 64x64

In [ ]:
compare_results(trained_models[(64, IMAGENETTE)], IMAGENETTE, 64)

##### Imagewoof Comparison 64x64

In [ ]:
compare_results(trained_models[(64, IMAGEWOOF)], IMAGEWOOF, 64)

##### Imagenette Comparison 128x128

In [ ]:
compare_results(trained_models[(128, IMAGENETTE)], IMAGENETTE, 128)

##### Imagewoof Comparison 128x128

In [ ]:
compare_results(trained_models[(128, IMAGEWOOF)], IMAGEWOOF, 128)

##### Imagenette Comparison 256x256

In [ ]:
compare_results(trained_models[(256, IMAGENETTE)], IMAGENETTE, 256)

##### Imagewoof Comparison 256x256

In [ ]:
compare_results(trained_models[(256, IMAGEWOOF)], IMAGEWOOF, 256)

#### Time trained

In [ ]:
def plot_time_trained(trained_models):
    if PLOT_TIME_TRAINED:
        time_trained = {
                # Key: model name: model parameters, for example: "nz=100_lrD=0.00010_lrG=0.00010_imgSize=64_uD=1_uG=1"
                # Value: [time trained imagenette, time trained imagewoof]
            }
        for img_size in IMG_SIZES:
            for database in DATABASES:
                for num, model in enumerate(trained_models[(img_size, database)].keys(), 1):
                    # Modelo referencia
                    for pair_model in trained_models[(img_size, database)][model]: # Modelo para cada clase
                        modelD_filename, modelG_filename, datasetTitle = pair_model

                        if os.path.isfile(modelD_filename) and os.path.isfile(modelG_filename):
                            patron = r"^(.*?)(?=_Model)"

                            coincidencia = re.search(patron, modelG_filename)
                            if coincidencia:
                                modelName = os.path.basename(coincidencia.group(1))
                                checkpoint = torch.load(modelG_filename)        
                                G_trainedTime = checkpoint['trainedTime']
                                G_trainedTime = sum(t for t in G_trainedTime if t is not None)
                                if modelName not in time_trained:
                                    time_trained[modelName] = [0, 0]
                                
                                if database == IMAGENETTE:
                                    time_trained[modelName][0] += G_trainedTime
                                elif database == IMAGEWOOF:
                                    time_trained[modelName][1] += G_trainedTime
                            else:
                                print("La cadena no coincide con el patrón.")
        # Datos
        modelos = []
        tiempo_imagenette = []
        tiempo_imagewoof = []
        for key, value in time_trained.items():
            modelos.append(key)
            # Convertir a minutos
            value[0] = value[0] / 60
            value[1] = value[1] / 60
            tiempo_imagenette.append(value[0])
            tiempo_imagewoof.append(value[1])

        print(len(modelos), len(tiempo_imagenette), len(tiempo_imagewoof))

        # Posiciones para las barras
        x = np.arange(len(modelos))
        ancho = 0.35  # ancho de las barras

        # Crear el plot
        fig, ax = plt.subplots(figsize=(12, 6))  # Aumenta el ancho, puedes ajustar el número
        ax.barh(x - ancho/2, tiempo_imagenette, height=ancho, label='Time Imagenette', color='cornflowerblue', align='center')
        ax.barh(x + ancho/2, tiempo_imagewoof, height=ancho, label='Time ImageWoof', color='indianred', align='center')

        # Etiquetas
        ax.set_xlabel('Time (min)')
        ax.set_title('Time trained (Minutes) Imagenette vs ImageWoof')
        ax.set_yticks(x)
        ax.set_yticklabels(modelos)
        ax.legend()

        plt.tight_layout()
        plt.show()

        print("Lowest time: ", min(tiempo_imagenette+tiempo_imagewoof))
        print("Highesh time: ", max(tiempo_imagenette+tiempo_imagewoof))

In [ ]:
plot_time_trained(trained_models)

In [ ]:
def dict_to_dataframe(data_dict, metrics, excel_filename=None, sheet_name='Sheet1'):
    """
    Convierte un diccionario en un DataFrame multiíndice tomando exactamente 3 decimales,
    guarda en Excel y subraya el valor máximo en cada columna.

    Parámetros:
    - data_dict (dict): Diccionario con métricas por base de datos y modelo.
    - metrics (list): Lista de nombres de métricas.
    - excel_filename (str, opcional): Nombre del archivo Excel para guardar.
    - sheet_name (str, opcional): Nombre de la hoja Excel.

    Retorna:
    - pd.DataFrame: DataFrame con el formato requerido.
    """
    if excel_filename:
        print("Saving a dict to ", excel_filename)

    databases = list(data_dict.keys())
    models = sorted({model for db in databases for model in data_dict[db]})

    data = []
    for model in models:
        row = []
        for db in databases:
            evaluations = data_dict[db].get(model, [None] * len(metrics))
            row.extend([float(f"{eval_metric:.3f}") if eval_metric is not None else None for eval_metric in evaluations])
        data.append(row)

    tuples = []
    for db in databases:
        tuples += [(db, metric) for metric in metrics]

    columns = pd.MultiIndex.from_tuples(tuples, names=["Databases", "Metric"])

    df = pd.DataFrame(data, columns=columns)
    df.insert(0, 'Metric', models)

    if excel_filename:
        df.to_excel(excel_filename, sheet_name=sheet_name)

        # Subrayar el máximo de cada columna
        wb = load_workbook(excel_filename)
        ws = wb[sheet_name]

        for col in range(2, ws.max_column + 1):  # Empezar desde la columna 2 para evitar 'Metric'
            max_val = float('-inf')
            max_cell = None

            for row in range(2, ws.max_row + 1):  # Comenzar en la fila 2 para evitar cabecera
                cell_value = ws.cell(row=row, column=col).value
                try:
                    cell_value = float(cell_value)
                    if cell_value > max_val:
                        max_val = cell_value
                        max_cell = ws.cell(row=row, column=col)
                except (TypeError, ValueError):
                    continue

            if max_cell:
                max_cell.font = Font(underline="single")
        wb.save(excel_filename)
    return df

___

## PRETAINED MODELS GANS

In [ ]:
torch.use_deterministic_algorithms(False)

# Diccionario de nombres de clases
CLASS_NAMES = {
    IMAGENETTE: [
        'tench', 'English springer', 'cassette player', 'chain saw',
        'church', 'French horn', 'garbage truck', 'gas pump',
        'golf ball', 'parachute'
    ],
    IMAGEWOOF: [
        'Australian terrier', 'Border terrier', 'Samoyed', 'Beagle',
        'Shih-Tzu', 'English foxhound', 'Rhodesian ridgeback',
        'Dingo', 'Golden retriever', 'Old English sheepdog'
    ]
}

# Diccionario con índices numéricos de las clases en ImageNet
IMAGENET_INDICES = {
    IMAGENETTE: {
        'tench': 0,
        'English springer': 217,
        'cassette player': 482,
        'chain saw': 491,
        'church': 497,
        'French horn': 566,
        'garbage truck': 569,
        'gas pump': 571,
        'golf ball': 574,
        'parachute': 701
    },
    IMAGEWOOF: {
        'Australian terrier': 193,
        'Border terrier': 182,
        'Samoyed': 258,
        'Beagle': 162,
        'Shih-Tzu': 155,
        'English foxhound': 167,
        'Rhodesian ridgeback': 159,
        'Dingo': 273,
        'Golden retriever': 207,
        'Old English sheepdog': 229
    }
}

# Diccionario con vectores one-hot (para BigGAN)
CLASS_VEC = {
    dataset: one_hot_from_names(CLASS_NAMES[dataset], batch_size=len(CLASS_NAMES[dataset]))
    for dataset in CLASS_NAMES
}

# Diccionario con tensores de índices numéricos (para StudioGAN)
CLASS_INDICES = {
    dataset: torch.tensor(
        [IMAGENET_INDICES[dataset][name] for name in CLASS_NAMES[dataset]], 
        dtype=torch.long
    )
    for dataset in CLASS_NAMES
}

TRUNCATION = 0.7
SHOW_EXAMPLES_PRETRAINED_GANS = False
EVALUATE_IMAGES_GANS = True

### Transforms

In [ ]:
TRANSFORM_VAL =transforms.Compose([
                transforms.Resize(224),
                transforms.CenterCrop(224),
                transforms.ToTensor(),
                transforms.Normalize(MEAN, STD),
])

TRANSFORM_TRAIN = transforms.Compose([
        # Recorte aleatorio escalado (manteniendo aspecto) a 224x224
        transforms.RandomResizedCrop(
            size=224, scale=(0.4, 1.0), ratio=(1.0, 1.0),
            interpolation=InterpolationMode.BICUBIC
        ),
        # Flip horizontal aleatorio
        transforms.RandomHorizontalFlip(p=0.5),
        # AutoAugment con la política de ImageNet
        transforms.AutoAugment(
            policy=AutoAugmentPolicy.IMAGENET,
            interpolation=InterpolationMode.BILINEAR
        ),
        # Variaciones aleatorias de color
        transforms.ColorJitter(brightness=0.2, contrast=0.2,
                                saturation=0.2, hue=0.02),
        # Blur
        transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.01, 0.4)),
        # Sharpennes
        transforms.RandomAdjustSharpness(sharpness_factor=1.5, p=0.5),  # Corregido aquí
        # Conversión a tensor [0,1] 
        transforms.ToTensor(),
        # (Opcional) Borrado aleatorio de un parche
        # transforms.RandomErasing(p=0.1),
        # Normalización con media y desvío estándar de ImageNet
        transforms.Normalize(MEAN, STD)
    ])

In [ ]:
def get_classes_from_model_name(model_name, database):
    """
    Devuelve el tensor correcto de clases dependiendo del tipo de modelo GAN.

    Args:
        model_name (str): Nombre del modelo GAN ('sagan studiogan', 'biggan', etc.).
        database (str): Base de datos ('IMAGENETTE' o 'IMAGEWOOF').

    Returns:
        Tensor: Tensor de clases apropiado para el modelo especificado.
    """
    model_name = model_name.lower()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    if model_name in ['sagan studiogan', 'sagan', 'sngan studiogan', 'sngan']:
        classes = CLASS_INDICES[database]
    elif model_name in ['biggan', 'biggan-deep-512']:
        classes = torch.from_numpy(CLASS_VEC[database])
    else:
        raise NotImplementedError(f'Unrecognized GAN type: {model_name}')

    return classes

def generar_imagenes_gan(modelo_gan, modelName, truncation=0.4):
    """
    Genera y muestra imágenes condicionales usando un modelo GAN preentrenado.

    Args:
        modelo_gan: Modelo GAN preentrenado (e.g. BigGAN, SAGAN).
        class_names: Lista de nombres de clases para generar imágenes.
        truncation: Factor de truncamiento del ruido (default 0.4).

    Returns:
        output: Tensor con imágenes generadas (batch_size, 3, H, W)
    """
    if SHOW_EXAMPLES_PRETRAINED_GANS:
        modelo_gan = copy.deepcopy(modelo_gan)
        # Mover al dispositivo (GPU si disponible)
        modelo_gan.to(device)

        for database in DATABASES:
            classes = get_classes_from_model_name(modelName, database).to(device)

            # Crear vector de ruido truncado
            noise_vec = truncated_noise_sample(truncation=truncation, batch_size=len(CLASS_NAMES[database]))
            # Convertir vectores a tensores de PyTorch
            noise_vec = torch.from_numpy(noise_vec)
            noise_vec = noise_vec.to(device)

            # Generar imágenes condicionales
            with torch.no_grad():
                output = modelo_gan(noise_vec, classes)

            # Mostrar imágenes generadas
            output_np = output.cpu().numpy()
            output_np = (output_np + 1) / 2  # Normalizar de [-1,1] a [0,1]

            fig, axes = plt.subplots(2, len(CLASS_NAMES[database])//2, figsize=(15, 6))
            axes = axes.flatten()
            for i, ax in enumerate(axes):
                img = np.transpose(output_np[i], (1, 2, 0))  # (H,W,3)
                ax.imshow(np.clip(img, 0, 1))
                ax.set_title(CLASS_NAMES[database][i])
                ax.axis('off')

            plt.suptitle(f'{modelName} Truncation: {truncation}')
            plt.tight_layout()
            plt.show()
            del classes, noise_vec
        del modelo_gan
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        gc.collect()

### Models

In [ ]:
G_BigGan = make_gan(gan_type='biggan')  # BigGAN preentrenado (256px)
G_BigGanDeep = make_gan(gan_type='biggan', model_name='biggan-deep-512')
G_Sagan = make_gan(gan_type='studiogan', model_name='SAGAN')  # asegúrate que el modelo es condicional
G_Sngan = make_gan(gan_type='studiogan', model_name='SNGAN')  # asegúrate que el modelo es condicional

pretrained_models = [
    (G_BigGan, 'BigGan'),
    (G_BigGanDeep, 'biggan-deep-512'),
    (G_Sagan, 'SAGAN'),
    (G_Sngan, 'SNGAN')
]

pretrained_modelsFromName = {
    'BigGan': G_BigGan,
    'biggan-deep-512': G_BigGanDeep,
    'SAGAN': G_Sagan,
    'SNGAN': G_Sngan,
}
print(f"CUDA memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
print(f"CUDA memory reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

### Images Examples

In [ ]:
generar_imagenes_gan(G_BigGanDeep, 'biggan-deep-512')

In [ ]:
generar_imagenes_gan(G_BigGan, 'BigGan')

In [ ]:
generar_imagenes_gan(G_Sagan, 'SAGAN StudioGAN')

In [ ]:
generar_imagenes_gan(G_Sngan, 'SNGAN StudioGAN') 

In [ ]:
print(f"CUDA memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
print(f"CUDA memory reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

### Image Creation Quality FID and IS

In [ ]:
def evaluate_gans_imgGeneration(pretrained_models, databases, paths_dataroot, truncation=0.4, num_batches=10, batch_size=32, device='cuda'):
    if EVALUATE_IMAGES_GANS:
        filename = os.path.join(PLOTS, 'GANsEvaluation.xlsx')

        if os.path.exists(filename):
            df = pd.read_excel(filename, header=[0, 1], index_col=0)

            # Eliminar columnas que se llamen 'Metric' si existen
            df = df.loc[:, df.columns.get_level_values(0) != 'Metric']

            print("Loaded evaluation from Excel:")
            return df

        allModelsEvaluation = {}
        for database in databases:
            allModelsEvaluation[database] = {}

            # Dataset y loader para imágenes reales
            real_dataset = ImageFolder(paths_dataroot[database]['val'], transform=TRANSFORM_VAL)
            loader_real = DataLoader(real_dataset, batch_size=batch_size, shuffle=False)

            for modelo_gan, modelName in pretrained_models:
                # Inicializar métricas
                fid_metric = FrechetInceptionDistance(feature=2048).to(device)
                is_metric = InceptionScore().to(device)

                # Procesar imágenes reales
                with torch.no_grad():
                    for imgs, _ in loader_real:
                        # Deshacer normalización
                        imgs_uint8 = copy.deepcopy(imgs)
                        for c in range(3):
                            imgs_uint8[:, c] = imgs_uint8[:, c] * STD[c] + MEAN[c]
                        imgs_uint8 = torch.clamp(imgs_uint8 * 255, 0, 255).type(torch.uint8)
                        fid_metric.update(imgs_uint8.to(device), real=True)

                # Función interna para generar imágenes sintéticas usando el GAN
                def generar_imagenes_gan():
                    modelo_gan.to(device)

                    classes = get_classes_from_model_name(modelName, database).to(device)
                    num_classes = len(classes)

                    # Repetir clases para ajustar al batch_size
                    classes_expanded = classes.repeat_interleave(batch_size // num_classes + 1, dim=0)[:batch_size]

                    noise_vec = truncated_noise_sample(truncation=truncation, batch_size=batch_size)
                    noise_vec = torch.from_numpy(noise_vec).to(device)

                    with torch.no_grad():
                        output = modelo_gan(noise_vec, classes_expanded)

                    output = (output + 1) / 2  # [-1,1] -> [0,1]
                    output = torch.clamp(output, 0, 1)

                    # Redimensionar correctamente con interpolación
                    output_resized = Functional.interpolate(output, size=(224, 244), mode='bilinear', align_corners=False)
                    output_resized = (output_resized * 255).type(torch.uint8)

                    del classes, noise_vec
                    torch.cuda.empty_cache()
                    torch.cuda.ipc_collect()
                    gc.collect()

                    return output_resized

                # Generar imágenes y actualizar métricas
                with torch.no_grad():
                    for _ in range(num_batches):
                        imgs_gen = generar_imagenes_gan()
                        fid_metric.update(imgs_gen.to(device), real=False)
                        is_metric.update(imgs_gen.to(device))

                # Calcular métricas finales
                fid_value = fid_metric.compute().item()
                is_mean, is_std = is_metric.compute()

                # Guardar resultados
                allModelsEvaluation[database][modelName] = [fid_value, float(is_mean), float(is_std)]

                # Opcionalmente, imprimir resultados
                print(f"Modelo: {modelName} | Base de datos: {database}")
                print(f"FID: {fid_value:.4f}")
                print(f"Inception Score: {float(is_mean):.4f} ± {float(is_std):.4f}\n")
                del modelo_gan, fid_metric, is_metric
                torch.cuda.empty_cache()
                gc.collect()

        dict_to_dataframe(allModelsEvaluation, ['fid', 'is_mean', 'is_std'], os.path.join(filename))

evaluate_gans_imgGeneration(pretrained_models, DATABASES, paths_dataroot)

In [ ]:
print(f"CUDA memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
print(f"CUDA memory reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

### Prepare Models for classifiers

___

# CLASSIFIERS

Imports

In [ ]:
torch.use_deterministic_algorithms(False)

BATCH_SIZE_CLASSIFIER = 50
NUM_EPOCHS_CLASSIFIER_IMAGENETTE = 300
NUM_EPOCHS_CLASSIFIER_IMAGEWOOF = 300
ROUNDS_PER_SAVE_MODELS_CLASSIFIER = 10

CLASSIFIERS_TRAIN = True

if CLASSIFIERS_TRAIN:
    SHOW_SAMPLES_DATASET = True
    CLASSIFIERS_TRAIN_REAL = True
    CLASSIFIERS_TRAIN_FAKE = True
    PLOT_ACCURACY_EVOLUTION_CLASSIFIERS = True
    PLOT_TIME_TRAINED_CLASSIFIERS = True
    CLASSIFIERS_ACCURACY = True
else:
    SHOW_SAMPLES_DATASET = False
    CLASSIFIERS_TRAIN_REAL = False
    CLASSIFIERS_TRAIN_FAKE = False
    PLOT_ACCURACY_EVOLUTION_CLASSIFIERS = False
    PLOT_TIME_TRAINED_CLASSIFIERS = False
    CLASSIFIERS_ACCURACY = False

SHOW_SAMPLES_DATASET = False



Parameters

### Dataset of 224x224

#### Validation Dataset

In [ ]:
class PreloadedImageMultipleFolder(Dataset):
    def __init__(self, root, id_to_index, transform=None):
        self.root = root
        self.transform = transform
        self.preloaded_data = []

        self.classes = [dir for dir in os.listdir(root) if os.path.isdir(os.path.join(root, dir))]

        for dir in self.classes:
            class_dir = os.path.join(root, dir)
            label = id_to_index[dir]
            # Obtener solo archivos de imagen en la carpeta
            self.samples = [os.path.join(class_dir, f) for f in os.listdir(class_dir) if f.lower().endswith(('jpg', 'jpeg', 'png'))]
            
            for path in self.samples:
                sample = Image.open(path).convert("RGB")  # Cargar imagen
                
                if self.transform:
                    sample = self.transform(sample)  # Aplicar la transformación
                
                self.preloaded_data.append((sample, torch.tensor(label)))  # Guardar imagen
            
    def __len__(self):
        return len(self.preloaded_data)

    def __getitem__(self, index):
        return self.preloaded_data[index]

def load_dataset_classifiers(dataroot, dataset_path, image_size, id_to_index):
    if os.path.exists(dataset_path):
        dataset = torch.load(dataset_path)
    else:
        print("Creating dataset for", os.path.basename(dataset_path))
        start_time = time.time()
        
        dataset = PreloadedImageMultipleFolder(root=dataroot, id_to_index=id_to_index,
                                transform=transforms.Compose([
                                    transforms.Resize(image_size),
                                    transforms.CenterCrop(image_size),
                                    transforms.ToTensor(),
                                    transforms.Normalize(MEAN, STD),
                                ]))

        # show_images_grid(dataset, dataloaderIW_64,"Training Images Imagewoof")
        print("Saving at: ", dataset_path)
        torch.save(dataset, dataset_path)
        print(f"\tTotal time creation of the dataset: {time.time() - start_time:.2f} seconds\n")
    return dataset

In [ ]:
def load_all_dataloaders_classifiers(img_size, batch_size, path_transformed_dataset, path_dataroot_imagewoof, path_dataroot_imagenette, val=False, num_workers=0):
    """
    Carga dataloaders por clase para Imagewoof2 e Imagenette2 en distintas resoluciones.

    Usa las variables globales IMAGEWOOF e IMAGENETTE para nombrar los datasets.

    Args:
        resolutions (list): Lista de resoluciones (por ejemplo [64, 128, 256]).
        batch_size (int): Tamaño del batch.
        path_transformed_dataset (str): Carpeta donde se guardan los datasets transformados.
        path_dataroot_imagewoof (str): Ruta al dataset original de Imagewoof2.
        path_dataroot_imagenette (str): Ruta al dataset original de Imagenette2.
        num_workers (int): Número de workers para los dataloaders.

    Returns:
        dict: Diccionario con dataloaders por dataset y resolución.
              Ej: {("imagewoof2", 64): {...}, ("imagenette2", 128): {...}, ...}
    """
    if val:
        dataloaders_path = os.path.join(path_transformed_dataset, "dataloaders_C_val.pt")
    else:
        dataloaders_path = os.path.join(path_transformed_dataset, "dataloaders_C.pt")

    if os.path.exists(dataloaders_path):
        dataloader_dicts = torch.load(dataloaders_path)
    else:
        dataloader_dicts = {}
        start_time = time.time()

        if val:
            addVal = "_val"
        else:
            addVal = ""

        # --- Imagenette ---
        dataset_dir = load_dataset_classifiers(path_dataroot_imagenette, os.path.join(TRANSFORMED_DATASET, f"{IMAGENETTE}_C_{img_size}{addVal}.pt"), img_size, id_to_index_imagenette)
        dataloader_dicts[IMAGENETTE] = DataLoader(dataset_dir, batch_size=batch_size, shuffle=True, num_workers=num_workers)

        # --- Imagewoof ---
        dataset_dir = load_dataset_classifiers(path_dataroot_imagewoof, os.path.join(TRANSFORMED_DATASET, f"{IMAGEWOOF}_C_{img_size}{addVal}.pt"), img_size, id_to_index_imagewoof)
        dataloader_dicts[IMAGEWOOF] = DataLoader(dataset_dir, batch_size=batch_size, shuffle=True, num_workers=num_workers)

        print("Saving at: ", path_transformed_dataset)
        torch.save(dataloader_dicts, dataloaders_path)
        print(f"\nTotal time creation of all the datasets: {time.time() - start_time:.2f} seconds")

    return dataloader_dicts


dataloader_classifiersVal = load_all_dataloaders_classifiers(
    img_size=224,
    batch_size=BATCH_SIZE_CLASSIFIER,
    path_transformed_dataset=TRANSFORMED_DATASET,
    path_dataroot_imagewoof=path_datarootImagewoofVal,
    path_dataroot_imagenette=path_datarootImagenetteVal,
    val=True
) 


In [ ]:
def denormalize(img_tensor):
    img = img_tensor.numpy().transpose((1, 2, 0))
    img = STD * img + MEAN
    img = np.clip(img, 0, 1)
    return img

def visualeDataloaderExamples(dataloader, database, images_per_class=2):
    inverse_imagenet_indices = {v: k for k, v in IMAGENET_INDICES[database].items()}
    class_names_sorted = sorted(inverse_imagenet_indices.values())
    images_collected = {class_name: [] for class_name in class_names_sorted}
    resize_transform = transforms.Resize((224, 224), antialias=True)

    for images_batch, labels in dataloader[database]:
        images_batch = resize_transform(images_batch)

        for img_tensor, label in zip(images_batch, labels):
            class_idx = CLASS_INDICES[database][label].item()
            class_name = inverse_imagenet_indices[class_idx]

            if len(images_collected[class_name]) < images_per_class:
                images_collected[class_name].append(img_tensor)

            if all(len(imgs) == images_per_class for imgs in images_collected.values()):
                break
        if all(len(imgs) == images_per_class for imgs in images_collected.values()):
            break

    num_classes = len(images_collected)
    cols = 10
    rows = images_per_class

    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = np.array(axes).reshape(rows, cols)

    class_idx = 0
    for col_idx, class_name in enumerate(class_names_sorted):
        imgs = images_collected[class_name]
        for row_idx in range(images_per_class):
            img_tensor = imgs[row_idx]
            img = denormalize(img_tensor)
            axes[row_idx, col_idx].imshow(img)
            axes[row_idx, col_idx].set_title(class_name if row_idx == 0 else "", fontsize=20)
            axes[row_idx, col_idx].axis('off')

    plt.tight_layout()
    plt.show()

#### Train Dataset

In [ ]:
""" train_transforms = transforms.Compose([
    # Recorte aleatorio escalado (manteniendo aspecto) a 224x224
    transforms.RandomResizedCrop(
        size=224, scale=(0.4, 1.0), ratio=(1.0, 1.0),
        interpolation=InterpolationMode.BICUBIC
    ),
    # Flip horizontal aleatorio
    transforms.RandomHorizontalFlip(p=0.5),
    # AutoAugment con la política de ImageNet
    transforms.AutoAugment(
        policy=AutoAugmentPolicy.IMAGENET,
        interpolation=InterpolationMode.BILINEAR
    ),
    # Variaciones aleatorias de color
    # Agresivo
    #transforms.ColorJitter(brightness=0.4, contrast=0.4,
    #                        saturation=0.4, hue=0.1),
    # Sutil
    transforms.ColorJitter(brightness=0.3, contrast=0.3,
                            saturation=0.2, hue=0.04),
    # Sharpennes
    transforms.RandomAdjustSharpness(sharpness_factor=1.5, p=0.5),                       
    # Conversión a tensor [0,1] 
    transforms.ToTensor(),
    # (Opcional) Borrado aleatorio de un parche
    # transforms.RandomErasing(p=0.1),
    # Normalización con media y desvío estándar de ImageNet
    transforms.Normalize(MEAN, STD)
])
 """
dataloader_classifiersTrain = {}

# Imagenette
train_data = ImageFolder(path_datarootImagenetteTrain, transform=TRANSFORM_TRAIN)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE_CLASSIFIER, shuffle=True,
                           num_workers=4, pin_memory=True, prefetch_factor=2, persistent_workers=True)
""" train_loader = DataLoader(train_data, batch_size=BATCH_SIZE_CLASSIFIER, shuffle=True,
                           num_workers=0) """
dataloader_classifiersTrain[IMAGENETTE] = train_loader

# Imagewoof
train_data = ImageFolder(path_datarootImagewoofTrain, transform=TRANSFORM_TRAIN)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE_CLASSIFIER, shuffle=True,
                           num_workers=4, pin_memory=True, prefetch_factor=2, persistent_workers=True)
""" train_loader = DataLoader(train_data, batch_size=BATCH_SIZE_CLASSIFIER, shuffle=True,
                           num_workers=0) """
dataloader_classifiersTrain[IMAGEWOOF] = train_loader

In [ ]:
# visualeDataloaderExamples(dataloader_classifiersTrain, IMAGENETTE, images_per_class=2)

In [ ]:
# visualeDataloaderExamples(dataloader_classifiersTrain, IMAGEWOOF, images_per_class=2)

In [ ]:
def show_sample_images(data_loader, database, num_images=8, mean=MEAN, std=STD):
    if SHOW_SAMPLES_DATASET:
        images, labels = next(iter(data_loader))
        num_images = min(num_images, len(images))

        fig, axes = plt.subplots(1, num_images, figsize=(num_images * 3, 3))

        mean = torch.tensor(mean).view(3, 1, 1)
        std = torch.tensor(std).view(3, 1, 1)

        for i in range(num_images):
            img = images[i]
            
            # Deshacer la normalización
            img = img * std + mean
            img = torch.clamp(img, 0, 1)

            img_np = img.permute(1, 2, 0).numpy()

            axes[i].imshow(img_np)
            axes[i].axis('off')
            axes[i].set_title(f'Label: {CLASS_NAMES[database][labels[i].item()]}')

        plt.show()


In [ ]:
show_sample_images(dataloader_classifiersTrain[IMAGENETTE], IMAGENETTE)

In [ ]:
show_sample_images(dataloader_classifiersTrain[IMAGEWOOF], IMAGEWOOF)

### Clasificators Models

In [ ]:
classifiersModels = {}

# Step 1: Initialize models
# model, lr
classifiersModels['Resnet34'] = [resnet34(weights=None, num_classes=10), 1e-4]
#classifiersModels['VGG13'] = [vgg13(weights=None, num_classes=10), 1e-4]
classifiersModels['MobileNet_v2'] = [mobilenet_v2(weights=None, num_classes=10), 5e-4]
classifiersModels['SqueezeNet1_1'] = [squeezenet1_1(weights=None, num_classes=10),  3e-4]
classifiersModels['AlexNet'] = [alexnet(weights=None, num_classes=10), 1e-4]

### Classifiers with real images

#### Train real Classifiers

In [ ]:
def save_classifier_model(model, optimizer, scheduler, epoch, best_epoch, losses, filename, trained_time, accuracy_list, accuracy_listVal):
    if scheduler is not None:
        scheduler = scheduler.state_dict()

    torch.save({
        'model_state_dict': model,
        'optimizer': optimizer,
        'scheduler_state_dict': scheduler,
        'epoch': epoch,
        'best_epoch': best_epoch,
        'losses': losses,
        'accuracy_list': accuracy_list,
        'accuracy_listVal': accuracy_listVal,
        'trainedTime': trained_time,
    }, filename)


def train_classifier_model(model_filename, model, optimizer, scheduler, dataloader, trained_epochs, num_epochs, device, best_epoch=None, dataloaderVal=None, num_classes=10, losses=None, accuracy_list=None, accuracy_listVal=None, trained_time=None):
    criterion = nn.CrossEntropyLoss()
    if losses is None:
        losses = [None] * num_epochs
    else:
        if len(losses) < num_epochs:
            losses.extend([None] * (num_epochs - len(losses)))

    if trained_time is None:
        trained_time = [None] * num_epochs
    else:
        if len(trained_time) < num_epochs:
            trained_time.extend([None] * (num_epochs - len(trained_time)))

    if accuracy_list is None:
        accuracy_list = [None] * num_epochs
    else:
        if len(accuracy_list) < num_epochs:
            accuracy_list.extend([None] * (num_epochs - len(accuracy_list)))

    if accuracy_listVal is None:
        accuracy_listVal = [None] * num_epochs
    else:
        if len(accuracy_listVal) < num_epochs:
            accuracy_listVal.extend([None] * (num_epochs - len(accuracy_listVal)))

    if best_epoch is not None:
        best_accuracy_val = accuracy_listVal[best_epoch]
    else:
        best_accuracy_val = 0
        best_epoch = 0
    model = model.to(device)
    model.train()
    accuracy = Accuracy(task="multiclass", num_classes=num_classes).to(device)
    

    for epoch in range(trained_epochs, num_epochs):
        running_loss = 0.0
        start_time = time.time()
        accuracy.reset()  # ← manual reset

        # Probar a borrar para comprobar que no afecta
        # Validation Accuracte
        if dataloaderVal is not None:
            with torch.no_grad():
                for i, data in enumerate(dataloaderVal):
                    images, labels = data
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    preds = torch.argmax(outputs, dim=1)
                    accuracy.update(preds, labels)
                epoch_accVal = accuracy.compute().item()
            accuracy.reset()  # ← manual reset
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        gc.collect()
        # Train
        for i, data in enumerate(dataloader):
            images, labels = data
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            with torch.no_grad():
                preds = torch.argmax(outputs, dim=1)
            accuracy.update(preds, labels)
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        gc.collect()
        with torch.no_grad():
            epoch_acc = accuracy.compute().item()
                
        
        epoch_loss = running_loss / len(dataloader)
        losses[epoch] = epoch_loss
        trained_time[epoch] = time.time() - start_time
        accuracy_list[epoch] = epoch_acc
        if dataloaderVal is not None:
            accuracy_listVal[epoch] = epoch_accVal
            accStr= f" Accuracy Val: {epoch_accVal:.4f} |"
        else:
            accStr = ""

        if ((torch.cuda.is_available() and ((torch.cuda.memory_allocated() / ((1024**2)) > 10000))) or ((torch.cuda.memory_reserved() / (1024**2)) > 10000)):
            print("************************************************************ WARNING ************************************************************")
            print(f"CUDA memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
            print(f"CUDA memory reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

        current_lr = optimizer.param_groups[0]['lr']
        # END OF EPOCH
        if scheduler is not None:
            scheduler.step()

        # Save the best validation possible, no possibility of overfitting
        if epoch_accVal > best_accuracy_val:

            best_accuracy_val = epoch_accVal
            best_epoch = epoch+1
            model = model.cpu()
            best_model_state_dict = copy.deepcopy(model.state_dict())
            model.to(device)
            best_model_values = [
                best_model_state_dict, # 0 model state
                copy.deepcopy(optimizer.state_dict()), # 1 optimizer
                copy.deepcopy(scheduler), # 2 scheduler
                None, # 3 epochs
                best_epoch, # 4 best_epoch
                None, # 5 losses
                None, # 6 model file name
                None, # 7 total trained time
                None, # 8 accuracy train list
                None # 9 accuracy val list
            ]
            print(f"\t\tBest version of model hit at epoch {epoch+1}, valAcc: {epoch_accVal}. parameters saved")
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            gc.collect()

        if (((epoch+1) % ROUNDS_PER_SAVE_MODELS_CLASSIFIER == 0) or ((epoch+1) == num_epochs)):
            best_model_values[3] = epoch+1
            best_model_values[5] = losses
            best_model_values[6] = model_filename
            best_model_values[7] = trained_time
            best_model_values[8] = accuracy_list
            best_model_values[9] = accuracy_listVal
            save_classifier_model(*best_model_values)
            
        print(f"\t[Epoch {epoch+1}/{num_epochs}] Loss: {epoch_loss:.4f} | Accuracy train: {epoch_acc:.4f} |{accStr}  LR actualizado: {current_lr:.8f} | Time: {trained_time[epoch]:.2f}s")


    print(f"\tModelo {model_filename} entrenado durante {num_epochs} épocas. Tiempo total: {sum(t for t in trained_time if t is not None):.2f}s")

def get_or_train_classifier_model(model_filename, model, lr, dataloader_classifiersTrain, dataloader_classifiersVal):
    # Model trained
    if IMAGENETTE in model_filename:
        dataloader = dataloader_classifiersTrain[IMAGENETTE]
        dataloaderVal = dataloader_classifiersVal[IMAGENETTE]
        num_epochs = NUM_EPOCHS_CLASSIFIER_IMAGENETTE
    elif IMAGEWOOF in model_filename:
        dataloader = dataloader_classifiersTrain[IMAGEWOOF]
        dataloaderVal = dataloader_classifiersVal[IMAGEWOOF]
        num_epochs = NUM_EPOCHS_CLASSIFIER_IMAGEWOOF

    if os.path.exists(model_filename):
        # Load the model and train if the epoch is less than NUM_EPOCHS_CLASSIFIER
        checkpoint = torch.load(model_filename)
        trained_epochs = checkpoint['epoch']
        trained_time = checkpoint['trainedTime']
        if trained_epochs >= num_epochs:
            print(f"\talready trained for {trained_epochs}/{num_epochs} epochs. Time spent: {sum(t for t in trained_time if t is not None):.2f}s. Skipping training.")
            print(f"\t\tBest epoch: {checkpoint['best_epoch']} hited Val: {checkpoint['accuracy_listVal'][checkpoint['best_epoch']]}, last training epoch hitted {checkpoint['accuracy_list'][trained_epochs-1]}")
        else:
            model.load_state_dict(checkpoint['model_state_dict'])
            model = model.to(device)
            optimizer = optim.Adam(model.parameters(), lr=lr)
            optimizer.load_state_dict(checkpoint['optimizer'])
            if checkpoint['scheduler_state_dict'] is not None:
                scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=lr/10)
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            else:
                scheduler = None
            losses = checkpoint['losses']
            accuracy_list = checkpoint['accuracy_list']
            accuracy_listVal = checkpoint['accuracy_listVal']
            trained_time = checkpoint['trainedTime']
            best_epoch = checkpoint['best_epoch']

            print(f"\talready trained for {trained_epochs}/{num_epochs} epochs.  Time spent: {sum(t for t in trained_time if t is not None):.2f}s. Best AccVal={accuracy_listVal[-1]} Continuing training.")


            train_classifier_model(model_filename, model, optimizer, scheduler, dataloader, trained_epochs, num_epochs, device, best_epoch=best_epoch, dataloaderVal=dataloaderVal, 
                                   num_classes=10, losses=losses, accuracy_list=accuracy_list, accuracy_listVal=accuracy_listVal, trained_time=trained_time)
    # Model not trained
    else:
        optimizer = optim.Adam(model.parameters(), lr=lr)
        scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=lr/10)
        scheduler = None # Quitar para prueba final
        print(f"\ttraining {model_filename} for {0}/{num_epochs} epochs")
        train_classifier_model(model_filename, model, optimizer, scheduler, dataloader, 0, num_epochs, device, dataloaderVal=dataloaderVal)


def train_all_classifiers(classifiersModels, dataloader_classifiersTrain, dataloader_classifiersVal, fake=False, truncation=None):
    trained_models_classifier = {}
    keys = set()
    for modelName in classifiersModels.keys():
        keys.add(modelName)

    keys = sorted(keys)
    print(keys)

    num = 1
    total = len(DATABASES) * len(classifiersModels)

    for database in DATABASES:
        print(f"*********************************************************************** {database} ***********************************************************************")
        for modelName in keys:
            print(f"Before Training [GPU] Used: {torch.cuda.memory_allocated() / 1024**2:.2f} MB | Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")
            print(f"\n\n{modelName}")
            model, lr = copy.deepcopy(classifiersModels[modelName])
            lr_str = "{:.6f}".format(lr)
            fake_str=""
            truncation_str=""
            if fake:
                fake_str="_fake"
                truncation_str=f"_t={truncation:.1f}"
                train = CLASSIFIERS_TRAIN_FAKE
            else:
                train = CLASSIFIERS_TRAIN_REAL
            
            if train:
                model_filename = os.path.join(MODELS_CLASSIFIERS, f"{modelName}_lr={lr_str}{truncation_str}_{database}{fake_str}.pt")
                print(model_filename)

                if database == IMAGENETTE:
                    num_epochs_classifier = NUM_EPOCHS_CLASSIFIER_IMAGENETTE
                elif database == IMAGEWOOF:
                    num_epochs_classifier = NUM_EPOCHS_CLASSIFIER_IMAGEWOOF
                
                """ if fake:
                    num_epochs_classifier += 200 """

                print(f"{num}/{total}. Training {modelName} for {num_epochs_classifier} epochs")
                get_or_train_classifier_model(model_filename, model, lr, dataloader_classifiersTrain, dataloader_classifiersVal)
                num += 1

                if database not in trained_models_classifier:
                    trained_models_classifier[database] = []
                if modelName not in trained_models_classifier[database]:
                    trained_models_classifier[database].append(model_filename)
                
                # Limpiar memoria de GPU
                del model, lr

                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
                gc.collect()
        # print(f"[GPU] Used: {torch.cuda.memory_allocated() / 1024**2:.2f} MB | Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")
    return trained_models_classifier

In [ ]:
trained_models_classifier = train_all_classifiers(classifiersModels, dataloader_classifiersTrain, dataloader_classifiersVal)

#### Accuracy

In [ ]:
# Función para plotear gráficos de barras agrupadas
def plot_accuracy_per_class_grouped(results, database, filename):
    class_labels = CLASS_NAMES[database]
    models = results[database].keys()
    num_models = len(models)

    x = np.arange(len(class_labels))
    width = 0.2

    plt.figure(figsize=(14, 7))

    for i, model in enumerate(models):
        accuracies = [results[database][model].get(c, 0) * 100 for c in range(10)]
        plt.bar(x + i * width, accuracies, width, label=model)

    plt.xlabel('Class')
    plt.ylabel('Precision (%)')
    plt.title(f'Precision per class of {database}')
    plt.xticks(x + width * (num_models - 1) / 2, class_labels, rotation=45, ha="right")
    plt.ylim(0, 100)
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
def get_bestModelClassifiers(models, trained_models_classifier, databases, dataloader_classifiersVal, dataloader_classifiersTrain, filename):
    if CLASSIFIERS_ACCURACY:
        bestModel = {}
        if os.path.exists(filename+'.xlsx'):
            # Cargar DataFrame (multiíndice en columnas, primera columna como índice de modelos)
            df = pd.read_excel(filename+'.xlsx', header=[0, 1], index_col=0)
            # Eliminar columnas que se llamen 'Metric' si existen
            df = df.loc[:, df.columns.get_level_values(0) != 'Metric']
            # Ahora puedes acceder sin errores
            bestModel = {}
            for database in df.columns.levels[0]:
                # Asegúrate que la columna existe en ambos niveles
                if (database, 'val-1') in df.columns:
                    val1_column = df[(database, 'val-1')]
                    best_model_name = val1_column.idxmax()
                    best_model_value = val1_column.max()
                    bestModel[database] = {
                        'Model': best_model_name,
                        'val-1': best_model_value
                    }
            # Mostrar DataFrame cargado y mejor modelo
            print(df)
            print(bestModel)
            return bestModel
        
        # Calculate best model
        num = 1
        total = len(databases) * len(models)
        allmodelsAccuracy = {}
        allmodelsAccuracyPerClass = {}

        for database in databases:
            dataloaderVal = dataloader_classifiersVal[database]
            dataloaderTrain = dataloader_classifiersTrain[database]

            bestModel[database] = None
            allmodelsAccuracy[database] = {}
            allmodelsAccuracyPerClass[database] = {}
            bestModelAcc = 0
            print(f'*************************** {database} ***************************')
            
            for modelName in models.keys():
                print(f"{num}/{total} - ModelName: {modelName} - {database}")
                allmodelsAccuracy[database][modelName] = {}
                allmodelsAccuracyPerClass[database][modelName] = {}

                for name in trained_models_classifier[database]:
                    name = os.path.basename(name)
                    if modelName in name:
                        path = os.path.join(MODELS_CLASSIFIERS, name)
                        if os.path.isfile(path):
                            
                            # Cargar modelo
                            checkpoint = torch.load(path)
                            model, lr = copy.deepcopy(models[modelName])
                            best_epoch = checkpoint['best_epoch']
                            model.load_state_dict(checkpoint['model_state_dict'])
                            model.to(device)
                            model.eval()

                            ### Por Clase
                            # Inicializar contadores
                            correct_per_class = defaultdict(int)
                            total_per_class = defaultdict(int)

                            total_correct = 0
                            total_samples = 0

                            with torch.no_grad():
                                for images, labels in dataloaderVal:
                                    images, labels = images.to(device), labels.to(device)
                                    outputs = model(images)
                                    preds = torch.argmax(outputs, dim=1)

                                    for label, pred in zip(labels, preds):
                                        label_id = label.item()
                                        total_per_class[label_id] += 1
                                        total_samples += 1
                                        if pred.item() == label_id:
                                            correct_per_class[label_id] += 1
                                            total_correct += 1

                            # Calcular precisión por clase
                            accuracy_per_class = {}
                            for class_id in total_per_class:
                                accuracy = correct_per_class[class_id] / total_per_class[class_id]
                                accuracy_per_class[class_id] = accuracy

                            # Mostrar precisión por clase
                            # for class_id in sorted(accuracy_per_class.keys()):
                                # print(f"Clase {class_id}: {accuracy_per_class[class_id]*100:.2f}% acierto")

                            # Mostrar precisión total
                            accuracy_total = total_correct / total_samples
                            # print(f"\nPrecisión total del modelo: {accuracy_total*100:.2f}%")
                            ###

                            # Inicializar métricas Top-1 y Top-5
                            accuracy_top1 = Accuracy(task="multiclass", num_classes=10, top_k=1).to(device)
                            accuracy_top5 = Accuracy(task="multiclass", num_classes=10, top_k=5).to(device)

                            ## Validation Accuracy
                            with torch.no_grad():
                                for images, labels in dataloaderVal:
                                    images, labels = images.to(device), labels.to(device)
                                    outputs = model(images)
                                    # Top-1
                                    preds_top1 = torch.argmax(outputs, dim=1)
                                    accuracy_top1.update(preds_top1, labels)
                                    # Top-5
                                    accuracy_top5.update(outputs, labels)  # outputs sin argmax

                            # Resultados finales
                            final_accuracyVal_top1 = accuracy_top1.compute().cpu().item()
                            final_accuracyVal_top5 = accuracy_top5.compute().cpu().item()
                            # Reset métricas antes del train
                            accuracy_top1.reset()
                            accuracy_top5.reset()

                            ## Train Accuracy
                            with torch.no_grad():
                                for images, labels in dataloaderTrain:
                                    images, labels = images.to(device), labels.to(device)
                                    outputs = model(images)
                                    preds_top1 = torch.argmax(outputs, dim=1)
                                    accuracy_top1.update(preds_top1, labels)
                                    accuracy_top5.update(outputs, labels)

                            final_accuracyTrain_top1 = accuracy_top1.compute().cpu().item()
                            final_accuracyTrain_top5 = accuracy_top5.compute().cpu().item()

                            print(f"\tAccuracy %:"
                                f"\n\t - Validation: Top-1: {final_accuracyVal_top1:.4f} | Top-5: {final_accuracyVal_top5:.4f}"
                                f"\n\t - Train:     Top-1: {final_accuracyTrain_top1:.4f} | Top-5: {final_accuracyTrain_top5:.4f}")
                            
                            allmodelsAccuracy[database][modelName] = [final_accuracyTrain_top1, final_accuracyVal_top1, final_accuracyTrain_top5, final_accuracyVal_top5, best_epoch]
                            allmodelsAccuracyPerClass[database][modelName] = accuracy_per_class
                            
                            if final_accuracyVal_top1 > bestModelAcc:
                                bestModelAcc = accuracy_total
                                bestModel[database] = [model.cpu(), name, bestModelAcc, accuracy_per_class]       
                            del model, lr
                num += 1
            print()
        
        print("Showing accuracy per class all models")
        print(allmodelsAccuracyPerClass)
        
        for database in databases:
            print(f"Best Model {database}:", bestModel[database][1], bestModel[database][2], bestModel[database][3])
            plot_accuracy_per_class_grouped(allmodelsAccuracyPerClass, database, filename+f'PerClass_{database}.png')

        dict_to_dataframe(allmodelsAccuracy, ['train-1', 'val-1', 'train-5', 'val-5', 'epoch'], filename+'.xlsx')
        print('Saved metrics')
        
        return bestModel

In [ ]:
bestModel = get_bestModelClassifiers(classifiersModels,trained_models_classifier, DATABASES, dataloader_classifiersVal, dataloader_classifiersTrain, os.path.join(PLOTS, 'ClassifiersAccuracyReal'))

In [ ]:
def plot_loss_evolution_classifiers(models, databases, trained_models_classifier):
    if PLOT_ACCURACY_EVOLUTION_CLASSIFIERS:
        num = 1
        for database in databases:
            plt.figure(figsize=(8, 4))
            fake_str = ""
            for modelName in models.keys():
                if "fake" in modelName:
                    fake_str = '_fake'
                for name in trained_models_classifier[database]:
                    name = os.path.basename(name)
                    if modelName in name:
                        path = os.path.join(MODELS_CLASSIFIERS, name)
                        if os.path.isfile(path):
                            # Cargar modelo
                            checkpoint = torch.load(path)
                            train_losses = checkpoint['losses']
                            plt.plot(train_losses, label=f"{modelName}")

            plt.title(f"{num}. {database} - Loss Evolution During Training", fontsize=16)
            plt.ylabel("Loss")
            plt.xlabel("Epoch")
            plt.legend()
            plt.tight_layout()
            # Save
            filename = os.path.join(PLOTS, f"ClassifiersLossEvolution_{database}{fake_str}.png")
            plt.savefig(filename, dpi=300)
            plt.show()

In [ ]:
plot_loss_evolution_classifiers(classifiersModels, DATABASES, trained_models_classifier)

In [ ]:
def plot_accuracy_evolution_classifiers(models, databases, trained_models_classifier):
    if PLOT_ACCURACY_EVOLUTION_CLASSIFIERS:
        
        num = 1
        for database in databases:
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True)
            fake_str = ""
            for modelName in models.keys():
                if "fake" in modelName:
                    fake_str = '_fake'
                for name in trained_models_classifier[database]:
                    name = os.path.basename(name)
                    if modelName in name:
                        path = os.path.join(MODELS_CLASSIFIERS, name)
                        if os.path.isfile(path):
                            
                            # Cargar modelo
                            checkpoint = torch.load(path)
                            train_accuracy = checkpoint['accuracy_list']
                            val_accuracy = checkpoint['accuracy_listVal']
                            ax1.plot(train_accuracy, label=f"{modelName}")
                            ax2.plot(val_accuracy, label=f"{modelName}")

            fig.suptitle(f"{num}. {database} - Accuracy Evolution During Training set vs Validation set", fontsize=16)

            ax1.set_title("Training set Accuracy")
            ax1.set_ylabel("Accuracy")
            ax2.set_xlabel("Epoch")
            ax1.legend()

            ax2.set_title("Validation set Accuracy")
            ax2.set_ylabel("Accuracy")
            ax2.set_xlabel("Epoch")
        
            ax2.legend()
            plt.ylim(0, 1)
            plt.tight_layout()
            # Save
            filename = os.path.join(PLOTS, f"ClassifiersAccuracyEvolution_{database}{fake_str}.png")
            plt.savefig(filename, dpi=300)
            plt.show()

In [ ]:
plot_accuracy_evolution_classifiers(classifiersModels, DATABASES, trained_models_classifier)

#### Time Trianed

In [ ]:
def plot_time_trained(models, trained_models_classifier):
    if PLOT_TIME_TRAINED_CLASSIFIERS:
        time_trained = {
                # Key: model name: model parameters, for example: "nz=100_lrD=0.00010_lrG=0.00010_imgSize=64_uD=1_uG=1"
                # Value: [time trained imagenette, time trained imagewoof]
            }
        num = 1
        fake_str = ""
        for database in DATABASES:
            for modelName in models.keys():
                if "fake" in modelName:
                    fake_str = '_fake'
                for name in trained_models_classifier[database]:
                    name = os.path.basename(name)
                    if modelName in name:
                        path = os.path.join(MODELS_CLASSIFIERS, name)
                        if os.path.isfile(path):
                            # Cargar modelo
                            checkpoint = torch.load(path)
                            trainedTime = checkpoint['trainedTime']
                            trainedTime = sum(t for t in trainedTime if t is not None)
                            if modelName not in time_trained:
                                time_trained[modelName] = [0, 0]
                            if database == IMAGENETTE:
                                time_trained[modelName][0] = trainedTime
                            elif database == IMAGEWOOF:
                                time_trained[modelName][1] = trainedTime
        # Datos
        modelos = []
        tiempo_imagenette = []
        tiempo_imagewoof = []
        for key, value in time_trained.items():
            modelos.append(key)
            # Convertir a minutos
            value[0] = value[0] / 60
            value[1] = value[1] / 60
            tiempo_imagenette.append(value[0])
            tiempo_imagewoof.append(value[1])

        # Posiciones para las barras
        x = np.arange(len(modelos))
        ancho = 0.35  # ancho de las barras

        # Crear el plot
        fig, ax = plt.subplots(figsize=(12, 6))  # Aumenta el ancho, puedes ajustar el número
        ax.barh(x - ancho/2, tiempo_imagenette, height=ancho, label='Tiempo Imagenette', color='cornflowerblue', align='center')
        ax.barh(x + ancho/2, tiempo_imagewoof, height=ancho, label='Tiempo ImageWoof', color='indianred', align='center')

        # Etiquetas
        ax.set_xlabel('Tiempo (min)')
        ax.set_title('Tiempo Entrenamiento (Minutos) imagenette vs ImageWoof')
        ax.set_yticks(x)
        ax.set_yticklabels(modelos)
        ax.legend()

        # Save
        filename = os.path.join(PLOTS, f"ClassifiersTimeTrained{fake_str}.png")
        plt.savefig(filename, dpi=300)
        plt.tight_layout()
        plt.show()

In [ ]:
plot_time_trained(classifiersModels, trained_models_classifier)

## Choose best DCGAN model

Calculate how many images per class 

In [ ]:
# total images in dataloader
for database in DATABASES:
    images_per_class = len(dataloader_classifiersVal[database])*BATCH_SIZE_CLASSIFIER // len(classifierFromName[database])
    print(len(dataloader_classifiersVal[database])*BATCH_SIZE_CLASSIFIER, '/', len(classifierFromName[database]) , '=', images_per_class)

IMAGES_PER_CLASS = 395

noise_for_accuracy = {}
for nz in NZ_VALUES:
    noise_for_accuracy[nz] = torch.randn(IMAGES_PER_CLASS, nz, 1, 1, device=device)

#### Accuracy - Fake batch

In [ ]:
class ImagenesGeneradasDataset(Dataset):
    def __init__(self, imgs, labels, transform):
        self.preloaded_data = []

        # Transform interno: Resize a 224x224 y conversión a tensor
        self.transform = transform

        for img, label in zip(imgs, labels):
            # img es tensor en este punto, conviértelo primero a PIL Image
            img_pil = to_pil_image(img)
            img_resized = self.transform(img_pil)
            self.preloaded_data.append((img_resized, label))

    def __len__(self):
        return len(self.preloaded_data)

    def __getitem__(self, idx):
        return self.preloaded_data[idx]

def getBestDCGANGeneratorsDCGAN(trained_models, classifierModel, trainedClassifiersFileName, filename):
    accuracy_fake_images = {
        # Key: database
        # value: dict{
        #   Key: image_Size
        #       value: dict{
        #           key: n (name of the classifier)
        #           value: [
        #               (modelG, lrD=0.01, lrG=0.02, % top 1 - right classificated images, % top 5)
        #           ]
        #       }
        #   }
        # }
    }
    # Initialization for no error in future functions
    bestModelForEachClass = {
        IMAGENETTE: {x:(-1,"") for x in classifierImagenette_inv},
        IMAGEWOOF:  {x:(-1, "") for x in classifierImagewoof_inv}
    }
    if CLASSIFIERS_ACCURACY and TRAIN_DCGAN:
        
        filenameImagenette = f'{os.path.join(PLOTS, filename)}_{IMAGENETTE}.xlsx'
        filenameImagewoof = f'{os.path.join(PLOTS, filename)}_{IMAGEWOOF}.xlsx'
        
        if os.path.exists(filenameImagenette) and os.path.exists(filenameImagewoof):
            print('Accuracy already calculated in', filename)
            return None

        total = 0
        for database in DATABASES:
            for img_size in IMG_SIZES:
                total += len(trained_models[(img_size, database)])
        print("Total Models: ", total)

        modelCount = 0
        for database in DATABASES:
            path = os.path.join(MODELS_CLASSIFIERS, trainedClassifiersFileName[database])
            print(path)
            if os.path.isfile(path):
                # Cargar modelo
                checkpoint = torch.load(path)
                classifierModel = copy.deepcopy(classifierModel)
                classifierModel.load_state_dict(checkpoint['model_state_dict'])
                classifierModel.to(device)
                classifierModel.eval()
            else:
                print("no se encontró el clasificador")
        
            print(f"DATABASE: {database}")
            if database not in accuracy_fake_images: # Iniciar el dict
                accuracy_fake_images[database] = {}
            for img_size in IMG_SIZES:
                print(f"\tIMAGE SIZE: {img_size}")
                if img_size not in accuracy_fake_images[database]:
                    accuracy_fake_images[database][img_size] = {}
                for num, model in enumerate(trained_models[(img_size, database)].keys(), 1):
                    # Modelo referencia
                    modelCount += 1
                    print(f"\t\tModel {num} - {modelCount}/{total}")
                    for pair_model in trained_models[(img_size, database)][model]: # Modelo para cada clase
                        modelD_filename, modelG_filename, datasetTitle = pair_model
                        if os.path.isfile(modelD_filename) and os.path.isfile(modelG_filename):
                            patron = r"^(.*?)(?=_Model)"
                            coincidencia = re.search(patron, modelG_filename)

                            if coincidencia:
                                modelName = os.path.basename(coincidencia.group(1))
                            else:
                                print("La cadena no coincide con el patrón.")
                                continue
                            
                            pattern = r"nz=(\d+)_lrD=([\d.]+)_lrG=([\d.]+)_imgSize=(\d+)_uD=(\d+)_uG=(\d+)_modData=(\d+)_ModelG-([a-z0-9]+)"
                            match = re.match(pattern, os.path.basename(modelG_filename))

                            if match:
                                nz = int(match.group(1))
                                lrD = float(match.group(2))
                                lrG = float(match.group(3))
                                imgSize = int(match.group(4))
                                uD = int(match.group(5))
                                uG = int(match.group(6))
                                modData = int(match.group(7))
                                n = match.group(8)
                            else:
                                print("La cadena no coincide con el patrón.")
                                continue

                            if imgSize != img_size:
                                print("ALGO RARO PASA")
                                continue
                            
                            if n not in accuracy_fake_images[database][img_size]:
                                accuracy_fake_images[database][img_size][n] = []

                            # Cargar modelo
                            netG = copy.deepcopy(generators_dict[imgSize][nz])
                            checkpoint = torch.load(modelG_filename)
                            netG.load_state_dict(checkpoint['model_state_dict'])

                            # Generar imagenes
                            netG = netG.to(device)
                            fakeImages = netG(noise_for_accuracy[nz]).detach().cpu()
                            fakeImages = (fakeImages + 1) / 2

                            if database == IMAGENETTE:
                                label = id_to_index_imagenette[n]
                            elif database == IMAGEWOOF:
                                label = id_to_index_imagewoof[n]
                            else:
                                print("ALGO RARO PASA")
                                continue
                            print(f"\t\t\tClase {label}")

                            labels = torch.full((IMAGES_PER_CLASS,), label, dtype=torch.long, device=device)

                            # Crear Dataset de cada clase para el modelo con estos parametros
                            dataset = ImagenesGeneradasDataset(fakeImages, labels, TRANSFORM_VAL)

                            # Crear Dataloader
                            dataloader = DataLoader(dataset, batch_size=BATCH_SIZE_CLASSIFIER, shuffle=True)

                            # Ver el % exito de clasificación del mejor modelo
                            # Inicializar métricas Top-1 y Top-5
                            accuracy_top1 = Accuracy(task="multiclass", num_classes=10, top_k=1).to(device)
                            accuracy_top5 = Accuracy(task="multiclass", num_classes=10, top_k=5).to(device)

                            ## Validation Accuracy
                            with torch.no_grad():
                                for images, labels in dataloader:
                                    images, labels = images.to(device), labels.to(device)
                                    outputs = classifierModel(images)

                                    # Top-1
                                    preds_top1 = torch.argmax(outputs, dim=1)
                                    accuracy_top1.update(preds_top1, labels)

                                    # Top-5
                                    accuracy_top5.update(outputs, labels)  # outputs sin argmax

                            # Resultados finales
                            final_accuracyVal_top1 = accuracy_top1.compute().item()
                            final_accuracyVal_top5 = accuracy_top5.compute().item()

                            modelG_filename = f"lrD={lrD}, lrG={lrG}, modData={modData}"

                            if bestModelForEachClass[database][n][0] < final_accuracyVal_top1:
                                #print(f"Updated accuracy for DB:{database} - class:{n}, from {bestModelForEachClass[database][n][0]} to {final_accuracyVal_top1}")
                                bestModelForEachClass[database][n] = (float(f"{final_accuracyVal_top1:.4f}"), modelG_filename)


                            print(f"\t\t\tAccuracy % - {classifierFromNameInv[database][n]} ({database}):"
                                f"\n\t\t\t\t- Validation: Top-1: {final_accuracyVal_top1:.4f} | Top-5: {final_accuracyVal_top5:.4f}")

                            accuracy_fake_images[database][img_size][n].append((modelG_filename, final_accuracyVal_top1, final_accuracyVal_top5))
                            del accuracy_top1, accuracy_top5
        print("Best Models: ")
        print(bestModelForEachClass)

        print("\nAll models")
        print(accuracy_fake_images)



        def create_dataframe(database_name, class_dict_inv, filename):
            data = {}
            def highlight_max(s):
                is_max = s == s.max()
                return ['text-decoration: underline;' if v else '' for v in is_max]
            
            for img_size, labels in accuracy_fake_images[database_name].items():
                for label, records in labels.items():
                    class_name = class_dict_inv[label]
                    for modelG_filename, acc_top1, _ in records:
                        if modelG_filename not in data:
                            data[modelG_filename] = {}
                        data[modelG_filename][class_name] = f'{acc_top1:.2f}'

            df = pd.DataFrame.from_dict(data, orient='index').reset_index().rename(columns={'index': 'Model_Config'})
            df_styled = df.style.apply(highlight_max, subset=pd.IndexSlice[:, df.columns[1:]])
            df_styled.to_excel(filename, index=False, engine='openpyxl')
            return df

        # Crear DataFrames separados
        create_dataframe(IMAGENETTE, classifierImagenette_inv, filenameImagenette)
        create_dataframe(IMAGEWOOF, classifierImagewoof_inv, filenameImagewoof)
            
    return bestModelForEachClass

#### Get Best DCGAN model for each class

In [ ]:
trainedClassifiersFileName = {IMAGENETTE: 'AlexNet_lr=0.000100_imagenette2.pt', IMAGEWOOF: 'AlexNet_lr=0.000100_imagewoof2.pt'}
model, lr = classifiersModels['AlexNet']
bestModelForEachClass = getBestDCGANGeneratorsDCGAN(trained_models, model, trainedClassifiersFileName, 'DCGANsClassifiersEvaluation')

In [ ]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
gc.collect()

## Train Classifiers with Fake images

### Create Dataset of best models generating fake images

In [ ]:
class ImagenesGeneradasDatasetDinamicForOneLabel(Dataset):
    def __init__(self, label, netG, nz, transform=TRANSFORM_TRAIN):
        self.netG = netG
        self.labels = torch.tensor(label, dtype=torch.long, device='cpu')
        self.nz = nz

        # Transform interno: Resize a 224x224 y conversión a tensor
        self.transform = transform
    
    def __len__(self):
        return IMAGES_PER_CLASS

    def __getitem__(self, idx):
        # Generar imagenes
        noise = torch.randn(1, nz, 1, 1, device='cpu')
        fakeImage = self.netG(noise).detach().cpu()
        fakeImage = (fakeImage + 1) / 2
        # Eliminar dimensión adicional (batch)
        fakeImage = fakeImage.squeeze(0)
        img_pil = to_pil_image(fakeImage)
        # resize a 224x224
        img_resized = self.transform(img_pil)
        return (img_resized, self.labels)
    

class ImagenesGeneradasDatasetDinamic(Dataset):
    def __init__(self, netG, modelName, database, transform, num_images=IMAGES_PER_CLASS, truncation=TRUNCATION):
        self.netG = netG
        self.truncation = truncation
        self.modelName = modelName.lower()
        self.database = database
        self.num_images = num_images
        self.transform = transform

    def __len__(self):
        return self.num_images
    

    def __getitem__(self, idx):
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

        if self.modelName in ['sagan studiogan', 'sagan', 'sngan studiogan', 'sngan']:
            chosen_class = torch.randint(len(CLASS_INDICES[self.database]), (1,), dtype=torch.long)
            class_label = CLASS_INDICES[self.database][chosen_class].to(device)
            label = chosen_class.item()

        elif self.modelName in ['biggan', 'biggan-deep-512']:
            chosen_index = np.random.choice(len(CLASS_VEC[self.database]), 1)
            class_label = torch.from_numpy(CLASS_VEC[self.database][chosen_index]).to(device)
            label = chosen_index.item()

        noise_vec = truncated_noise_sample(truncation=self.truncation, batch_size=1)
        noise_vec = torch.from_numpy(noise_vec).to(device)

        self.netG.to(device)
        with torch.no_grad():
            output = self.netG(noise_vec, class_label)

        image = output.cpu().squeeze(0)
        image = (image + 1) / 2  # [-1,1] -> [0,1]
        image = transforms.ToPILImage()(image)
        if self.transform:
            image = self.transform(image)

        return image, label


def createFakeDataloadersDCGAN(bestModelForEachClass):
    dataloader_classifiersTrainFake = {}
    for database in DATABASES:
        datasetsFakeImages = []
        for n in classifierFromNameInv[database]:
            # Mejor modelo generando imagenes de esta clase
            modelG_filename = bestModelForEachClass[database][n][1]
            if os.path.isfile(modelG_filename):
                pattern = r"nz=(\d+)_lrD=([\d.]+)_lrG=([\d.]+)_imgSize=(\d+)_uD=(\d+)_uG=(\d+)_modData=(\d+)_ModelG-([a-z0-9]+)"
                match = re.match(pattern, os.path.basename(modelG_filename))

                if match:
                    nz = int(match.group(1))
                    lrD = float(match.group(2))
                    lrG = float(match.group(3))
                    imgSize = int(match.group(4))
                    uD = int(match.group(5))
                    uG = int(match.group(6))
                    modData = int(match.group(7))
                    n = match.group(8)
                else:
                    print("La cadena no coincide con el patrón.")
                    continue

                netG = copy.deepcopy(generators_dict[imgSize][nz])
                checkpoint = torch.load(modelG_filename)
                netG.load_state_dict(checkpoint['model_state_dict'])

                if database == IMAGENETTE:
                    label = id_to_index_imagenette[n]
                elif database == IMAGEWOOF:
                    label = id_to_index_imagewoof[n]
                else:
                    print("ALGO RARO PASA")
                    continue
                #print(f"Clase {label}")

                # Crear Dataset de cada clase para el modelo con estos parametros
                datasetsFakeImages.append(ImagenesGeneradasDatasetDinamicForOneLabel(label, netG, nz, TRANSFORM_TRAIN))

        # All dataloaders ready
        dataset_concatenado = ConcatDataset(datasetsFakeImages)
        dataloader_classifiersTrainFake[database] = DataLoader(dataset_concatenado, batch_size=BATCH_SIZE_CLASSIFIER, shuffle=True)

    return dataloader_classifiersTrainFake

def createFakeDataloader(model, modelName, truncation):
    dataloader_classifiersTrainFake = {}
    for database in DATABASES:
        datasetsFakeImages = ImagenesGeneradasDatasetDinamic(model, modelName, database, TRANSFORM_TRAIN, truncation=truncation)
        # All dataloaders ready
        dataloader_classifiersTrainFake[database] = DataLoader(datasetsFakeImages, batch_size=BATCH_SIZE_CLASSIFIER, shuffle=True)
    return dataloader_classifiersTrainFake

model, modelName = pretrained_models[0]
print(model, modelName)
dataloader_classifiersTrainFake_t07 = createFakeDataloader(model, modelName, truncation=0.7)
#dataloader_classifiersTrainFake = createFakeDataloadersDCGAN(bestModelForEachClass)

In [ ]:
visualeDataloaderExamples(dataloader_classifiersTrainFake_t07, IMAGENETTE, images_per_class=4)

In [ ]:
visualeDataloaderExamples(dataloader_classifiersTrainFake_t07, IMAGEWOOF, images_per_class=4)

### Train Classifiers with fake images

truncation = 0.7

In [ ]:
DATABASES = [IMAGEWOOF]
classifiersModels['VGG13'] = [vgg13(weights=None, num_classes=10), 1e-4]

In [ ]:
trained_models_classifierFake = train_all_classifiers(classifiersModels, dataloader_classifiersTrainFake_t07, dataloader_classifiersVal, fake=True, truncation=0.7)

### Get the accuracy on real images of the classifators trained with fake images

In [ ]:
bestModel = get_bestModelClassifiers(classifiersModels, trained_models_classifierFake, DATABASES, dataloader_classifiersVal, dataloader_classifiersTrainFake_t07, os.path.join(PLOTS, 'ClassifiersAccuracyFake'))

In [ ]:
plot_loss_evolution_classifiers(classifiersModels, DATABASES, trained_models_classifierFake)

In [ ]:
plot_accuracy_evolution_classifiers(classifiersModels, DATABASES, trained_models_classifierFake)

### Time trained

In [ ]:
plot_time_trained(classifiersModels, trained_models_classifierFake)